# MUSANGA V3.1 · NB06 — PlanetScope & embedding spatial information

**Question:** Does fine-scale spatial context add meaningful and transferable information for estimating *Musanga* fraction beyond the established Sentinel-2 and satellite-embedding baseline?

**Stage 1 — exploration, not model fitting:** establish native resolution, available 4-band Planet annual/monthly imagery, annual 9-band NIR texture, 64-band 10 m embeddings and the existing 100 m UAV target cells. Display matched Planet and embedding patches, inspect 30/50/100 m supervision availability and spatial alignment. Compare native PlanetScope canopy texture with spatial variability in embeddings without confusing apparent training fit with generalization.

**Stage 2 — planned ablation, only after patch QA:** model comparison under identical targets and folds:
- **B0:** S2 annual + S2 quarterly + annual satellite embeddings (NB04B baseline).
- **B1:** B0 + handcrafted Planet texture (existing NB05 features).
- **B2:** B0 + multiscale **spatial texture of embeddings** (not merely their 64 central-pixel values).
- **B3:** B0 + Planet spatial CNN features.
- **B4:** B0 + embedding spatial CNN features.
- **B5:** B0 + both Planet and embedding CNN features.

First pilot at 50 m, then 30/100 m. **Same continuous UAV target**, same spatially held-out folds for within-site testing and leave-one-forest-out for geographic transfer. Hold out complete spatial areas/sites *before* patch extraction or learned representation training. Run supervised CNN embedding extraction out-of-fold, and fit all scalers/PCA/normalization from training folds only.

**Constraints:** Annual Planet image is 4.77 m and **may not be cloud-free**. Satellite embeddings are 10 m and already summarize S1/S2 information. Yoko Planet 2021 vs UAV 2020 is a known mismatch. Existing NB04B 100 m samples are used for preview only; no CNN trained in this first stage. No full-ROI 4.77 m imagery is assumed downloaded.


## 00 · Settings — exploratory preview only


In [ ]:
from pathlib import Path
FINAL=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                          Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
if FINAL is None:
    from google.colab import drive
    drive.mount('/content/drive')
    FINAL=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                              Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert FINAL and FINAL.is_dir(),'Musanga Final workspace missing'
ROOT=FINAL/'Outputs/V3_1'
NB05=ROOT/'05_sat_predictor_exploration'
NB04=ROOT/'04_satellite_predictors/input_layers'
NB04B=ROOT/'04B_roi_quarterly_LOSO'
OUT=ROOT/'06_spatial_planet_embeddings'
FIG=OUT/'figures'
for d in (OUT,FIG):d.mkdir(parents=True,exist_ok=True)
SITES=['Baego','MOS','Yangambi','Yoko']
PLANET_YEAR={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2021}
EMBEDDING_YEAR={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2020}
PREVIEW_PER_SITE=3
PREVIEW_PATCH_PIXELS=48      # ~229m Planet context for each selected 100m cell
EMB_PATCH_PIXELS=23          # ~230m at 10m embedding resolution
RANDOM_SEED=20261024
print('Workspace:',FINAL)


## 01 · Validate source products and review temporal/resolution limitations

Checks actual files, expected 19-band annual Planet, 9-band NIR texture and the existing 64-band annual embeddings. Annual Planet composition bands are:
1–4 screened mean B/G/R/N; 5–8 screened median; 9–12 raw mean; **13–16 raw median**; 17–19 raw valid count / screened valid count / number screened out.
The screened product is **heuristic only**; use the raw median as the stable first CNN baseline.

NB04B's original 100 m sample cache is *not recreated* or overwritten.


In [ ]:
import numpy as np,pandas as pd,rasterio,matplotlib.pyplot as plt
from rasterio.windows import Window
from rasterio.warp import transform as rio_transform
from rasterio.enums import Resampling
from pyproj import Transformer
from IPython.display import display

def planet_file(site,kind):
    root=NB05/('derived_features' if kind=='Texture' else 'input_layers')
    folder=f'Musanga_V31_NB05_UAV_ROI_{kind}'
    tag=f'NB05_UAV_ROI_{site}_{PLANET_YEAR[site]}_'
    pattern=tag+('Annual_19band*.tif' if kind=='Annual' else 'Texture_9band*.tif')
    matches=sorted((root/folder).glob(pattern))
    if len(matches)!=1:
        raise FileNotFoundError(f'{site} {kind}: expected one file, found {len(matches)}: {matches}')
    return matches[0]

def embed_file(site):
    return NB04/'EMB_ANNUAL'/f'{site}_EMB_ANNUAL_{EMBEDDING_YEAR[site]}_10m.tif'

AUDIT=[]
for site in SITES:
    for family,filepath,expected in (
        ('PLANET_ANNUAL',planet_file(site,'Annual'),19),
        ('PLANET_TEXTURE',planet_file(site,'Texture'),9),
        ('EMB_ANNUAL',embed_file(site),64)):
        assert filepath.is_file(),filepath
        with rasterio.open(filepath) as src:
            assert src.count==expected,(site,family,'expected',expected,'found',src.count)
            assert src.crs and src.width>0 and src.height>0
            AUDIT.append(dict(site=site,family=family,year=PLANET_YEAR[site] if family!='EMB_ANNUAL' else EMBEDDING_YEAR[site],
                bands=src.count,pixel_size_m=round(abs(src.res[0]),3),
                width=src.width,height=src.height,crs=str(src.crs),
                size_MB=round(filepath.stat().st_size/1e6,2),path=str(filepath)))
AUDIT=pd.DataFrame(AUDIT)
display(AUDIT.drop(columns=['path']))
AUDIT.to_csv(OUT/'01_input_audit.csv',index=False)
assert (NB05/'07_final_input_audit/07_site_summary.csv').exists(),'Complete NB05 Section 07 audit first'
print('All 12 native products found: four forests × annual Planet/texture/embeddings.')
print('Note: The 10m embeddings are not 4.77m imagery; do not upscale them to claim new spatial detail.')


## 02 · Select existing 100 m UAV reference cells (identical NB04B targets)

Only select example cells; avoid random point selection from the full Planet image, which would include areas without UAV supervision. Sampling is stratified by low/medium/high Musanga abundance **within each site**, so each forest's previews show more than zero-cover background. Cell centers are computed from NB04B's exact S2-grid transform, not from row/col as if they were Planet raster indices.


In [ ]:
from affine import Affine
SAMPLE_PATH=NB04B/'samples/roi_cells_100m.parquet'
assert SAMPLE_PATH.is_file(),SAMPLE_PATH
DF=pd.read_parquet(SAMPLE_PATH)
assert {'site','row','col','mean_probability'}.issubset(DF.columns)
DF=DF.loc[DF.site.isin(SITES)].copy()
assert len(DF)>0
RNG=np.random.default_rng(RANDOM_SEED)
CENTERS={}
SELECTED=[]
for site in SITES:
    local=DF.loc[DF.site==site].copy()
    assert not local.empty,site
    # Existing 100m NB04B samples use the 10m S2 affine aggregated exactly 10×.
    raw=NB04/'RAW_ANNUAL'/f'{site}_RAW_ANNUAL_{EMBEDDING_YEAR[site]}_10m.tif'
    assert raw.is_file(),raw
    with rasterio.open(raw) as ds:
        assert abs(ds.res[0]-10)<.05 and abs(ds.res[1]-10)<.05
        tfm=ds.transform*Affine.scale(10,10)
        crs=ds.crs
    for i in range(PREVIEW_PER_SITE):
        ordered=local.sort_values('mean_probability')
        idx=int(round((i+.5)*(len(ordered)-1)/PREVIEW_PER_SITE))
        sample=ordered.iloc[idx]
        cx,cy=tfm*(float(sample.col)+.5,float(sample.row)+.5)
        SELECTED.append(dict(site=site,group=str(sample.group),mosaic=str(sample.mosaic),
            fraction=float(sample.mean_probability),row=int(sample.row),col=int(sample.col),
            center_x=float(cx),center_y=float(cy),crs=str(crs)))
    CENTERS[site]=dict(transform=tfm,crs=crs)
PREVIEW=pd.DataFrame(SELECTED)
display(PREVIEW[['site','group','fraction','row','col']].round(3))
PREVIEW.to_csv(OUT/'02_preview_cells.csv',index=False)
print('Preview targets are UAV-model estimates at 100 m, NOT independent field labels.')


## 03 · Matched *native* Planet RGB/NIR and embedding-neighborhood previews

For the same UAV-target cell center, read a ~229 m **Planet raw annual median RGB patch** and a ~230 m **10 m satellite-embedding patch**. Embeddings contain 64 latent bands; visualize the first three PCA components as an exploratory false-colour composite (PCA fit only on the displayed patch, **not** for any prediction model). Preserve image pixel scales. This step helps spot spatial shifts, boundary masking, and structural patterns.

The common center is transformed to each raster's CRS if necessary. If a patch crosses a source footprint boundary it is padded with zeros and flagged by nonzero-band masks; no fabricated reflectance values.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
def native_patch(path,cx,cy,center_crs,size,bands):
    with rasterio.open(path) as ds:
        trans=Transformer.from_crs(center_crs,ds.crs,always_xy=True)
        x,y=trans.transform(cx,cy)
        cc,rr=~ds.transform*(x,y)
        start_col=int(np.floor(cc-size/2))
        start_row=int(np.floor(rr-size/2))
        w=Window(start_col,start_row,size,size)
        arr=ds.read(bands,window=w,boundless=True,fill_value=0).astype('float32')
        return arr,ds.crs,ds.res
def rgb_fixed(arr,valid):
    # Input arr is RGB band order; calculate one robust stretch per local patch.
    if np.count_nonzero(valid)<10:return np.ones((*valid.shape,3),dtype='float32')
    p=arr[:,valid]
    lo=np.percentile(p,2,axis=1)
    hi=np.percentile(p,98,axis=1)
    rgb=np.moveaxis(np.clip((arr-lo[:,None,None])/(hi[:,None,None]-lo[:,None,None]+1e-6),0,1),0,-1)
    rgb[~valid]=1
    return rgb
def embed_pca_rgb(cube):
    mask=np.isfinite(cube).all(axis=0)&np.any(cube!=0,axis=0)
    out=np.ones((*mask.shape,3),dtype='float32')
    if mask.sum()<10:return out,mask
    X=cube[:,mask].T
    X=StandardScaler().fit_transform(X)
    vals=PCA(n_components=3).fit_transform(X)
    pct_lo=np.percentile(vals,2,axis=0);pct_hi=np.percentile(vals,98,axis=0)
    out[mask]=np.clip((vals-pct_lo)/(pct_hi-pct_lo+1e-6),0,1)
    return out,mask

fig,axes=plt.subplots(len(PREVIEW),2,figsize=(8,3.0*len(PREVIEW)),layout='constrained')
rows=[]
for i,row in enumerate(PREVIEW.itertuples(index=False)):
    pa=planet_file(row.site,'Annual')
    eb=embed_file(row.site)
    # Planet annual 13-16 = B,G,R,N raw median; select RGB order 15,14,13
    p,_,pres=native_patch(pa,row.center_x,row.center_y,row.crs,PREVIEW_PATCH_PIXELS,[15,14,13])
    e,_,eres=native_patch(eb,row.center_x,row.center_y,row.crs,EMB_PATCH_PIXELS,list(range(1,65)))
    pmask=np.all(np.isfinite(p)&(p>0),axis=0)
    eviz,emask=embed_pca_rgb(e)
    axes[i,0].imshow(rgb_fixed(p,pmask));axes[i,1].imshow(eviz)
    axes[i,0].set_title(f'{row.site} | UAV {row.fraction*100:.1f}% | Planet RGB')
    axes[i,1].set_title('10 m embedding neighborhood (local PCA)')
    for ax in axes[i]:ax.axis('off')
    rows.append(dict(site=row.site,fraction=row.fraction,
                     planet_valid_pct=100*pmask.mean(),embedding_valid_pct=100*emask.mean(),
                     planet_pixel_size_m=abs(pres[0]),embedding_pixel_size_m=abs(eres[0])))
plt.savefig(FIG/'03_planet_vs_embedding_native_patches.png',dpi=140,bbox_inches='tight')
plt.show()
PATCH_QA=pd.DataFrame(rows)
display(PATCH_QA.round(2))
PATCH_QA.to_csv(OUT/'03_patch_coverage.csv',index=False)
print('IMPORTANT: local PCA colours are arbitrary across panels/sites and cannot be compared spectrally.')


## 04 · Spatial embeddings vs Planet: first quantitative exploratory features

To assess whether embeddings also contain spatial texture, summarize **within-patch heterogeneity** of their 64 channels. Compare each 100 m target cell's:
- Planet NIR local texture (from NB05's existing annual raster, sampled as local median);
- embedding spatial heterogeneity (median channelwise SD over a ~230 m context), plus embedding central-context mean differences.

**Descriptive only:** 12 deliberately stratified preview cells are too few for useful correlations or model fitting. This diagnostic checks that features are finite and nontrivial. Later we will extract these features across *all* labelled cells, use consistent physical windows (not just fixed pixel counts), and evaluate added value on geographic holdouts.


In [ ]:
from scipy.ndimage import median_filter
QA=[]
for row in PREVIEW.itertuples(index=False):
    e,_,_=native_patch(embed_file(row.site),row.center_x,row.center_y,row.crs,EMB_PATCH_PIXELS,list(range(1,65)))
    m=np.isfinite(e).all(axis=0)&np.any(e!=0,axis=0)
    if m.sum()<25:continue
    em=e[:,m]
    vari=np.std(em,axis=1)
    center=e[:,EMB_PATCH_PIXELS//2,EMB_PATCH_PIXELS//2]
    context=np.mean(em,axis=1)
    t,_,_=native_patch(planet_file(row.site,'Texture'),row.center_x,row.center_y,row.crs,
                       PREVIEW_PATCH_PIXELS,[1,2,3,4])
    tm=np.isfinite(t).all(axis=0)&(t>=0).all(axis=0)&np.any(t>0,axis=0)
    QA.append(dict(site=row.site,uav_fraction=row.fraction,
        embedding_channel_sd_median=float(np.median(vari)),
        embedding_context_difference=float(np.mean(abs(center-context))),
        planet_NIR_CV_r3_median=float(np.median(t[1,tm])) if tm.any() else np.nan,
        valid_embedding_pct=100*m.mean(),valid_planet_texture_pct=100*tm.mean()))
Q=pd.DataFrame(QA)
display(Q.round(4))
Q.to_csv(OUT/'04_spatial_feature_feasibility.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(10,4),layout='constrained')
for site,group in Q.groupby('site'):
    axes[0].scatter(group.uav_fraction*100,group.embedding_channel_sd_median,label=site)
    axes[1].scatter(group.uav_fraction*100,group.planet_NIR_CV_r3_median,label=site)
axes[0].set(xlabel='UAV Musanga fraction (%)',ylabel='Median channel SD in embedding patch',
            title='Embeddings: spatial heterogeneity')
axes[1].set(xlabel='UAV Musanga fraction (%)',ylabel='Planet NIR CV (radius 3)',
            title='Planet: native-scale texture')
axes[1].legend()
plt.savefig(FIG/'04_spatial_heterogeneity_feasibility.png',dpi=150,bbox_inches='tight')
plt.show()
print('This plot has only 12 stratified examples. Do NOT interpret it as prediction skill.')


## 05 · Decision gate and forthcoming modelling experiment

Before CNN model fitting, confirm:
1. Patch-validity percentages and visual alignment of Planet with its target cells; independent UAV orthomosaic/probability overlay will follow when locating the original teacher rasters.
2. Check that Planet **2021** in Yoko is an intentional temporal mismatch against UAV **2020**.
3. Use 50 m targets first and compare 30/100 m; target extraction must use the original **full-resolution UAV probability rasters**, avoiding interpolation of 100 m labels.
4. Train compact models with group/spatial folds: **100 m/50 m patches overlap heavily**, so randomly split cells is not an acceptable test.
5. Compare `B0 → B1/B2 → B3/B4 → B5` using the same site holdout and identical target cells; summarize MAE (pp), RMSE (pp), R², high-cover bias, calibration, and performance separately at MOS. Aggregate CNN output to target cells fairly.

**NB06 currently implements the exploratory entry point (Sections 00–04) and experimental specification; CNN training follows after visual QA.** Avoid building a large training architecture before verifying supervision and patch geometry.


## 06 · Verify original UAV probability rasters and Planet co-registration

The authoritative teacher rasters are the same **M2 warmstart full-mosaic U-Net probability GeoTIFFs** used by NB04 (not the 100 m aggregate). We inspect their CRS, footprint, NoData encoding, dynamic range, and cross-check the reconstructed 100 m values against NB04B sampled targets for each mosaic. The code automatically tests several plausible uint16 denominators but **refuses to accept a scale** unless the agreement is good. Do not retrain anything if this fails.

Generate site/mosaic **side-by-side Planet RGB, original UAV orthomosaic RGB, and continuous UAV probability** panels with one common geographic extent at native pixels. The images may have different sensor dates and radiometry. The purpose is to check obvious geometric shifts, not to infer identity from colours.

*Important*: UAV 2020 and Planet 2021 for Yoko; one-year mismatch is logged. A visually plausible image is not proof of subpixel registration; if the overlay looks shifted, fix georeferencing upstream before extracting labels.


In [ ]:
# 06 — Authoritative UAV teacher paths and raster metadata
import numpy as np,pandas as pd,rasterio,matplotlib.pyplot as plt, json,math
from rasterio.warp import reproject,Resampling,transform_bounds
from rasterio.windows import from_bounds,Window
from rasterio.features import rasterize
from affine import Affine
from shapely.geometry import box
from IPython.display import display

UAV_TEACHERS=FINAL/'Outputs/V3_1/03_M2_warmstart_c6a33504c37efb2ebb84_full_mosaic/probability_maps'
UAV_MOSAICS={
 'Baego':['Baego.tif'],
 'MOS':['Mos_1_2_transparent_mosaic_group1_georef.tif','Mos_5_6_transparent_mosaic_group1_georef.tif'],
 'Yangambi':['Yangambi_georef.tif'],
 'Yoko':['Yoko_kvo_georef.tif']
}
MANIFEST_FILE=FINAL/'Outputs/V3_1/00_input_audit/00_input_manifest.json'
assert MANIFEST_FILE.exists(),MANIFEST_FILE
uav_manifest=json.loads(MANIFEST_FILE.read_text())
MOSAIC_PATH={x['file']:Path(x['path']) for x in uav_manifest['mosaics']}
for site,names in UAV_MOSAICS.items():
    for name in names:
        assert name in MOSAIC_PATH and MOSAIC_PATH[name].is_file(), ('Missing orthomosaic',site,name)
def teacher_file(name):
    return UAV_TEACHERS/f'{Path(name).stem}_M2_warmstart_epoch02_probability_u16_FULL.tif'
SOURCE_TABLE=[]
for site,names in UAV_MOSAICS.items():
    for name in names:
        path=teacher_file(name)
        assert path.is_file(),f'Authoritative UAV model output missing: {path}'
        with rasterio.open(path) as ds:
            assert ds.count>=1 and ds.crs and ds.width>0 and ds.height>0
            # Distributed window samples for a rapid encoding diagnostic.
            samples=[]
            for frac_y,frac_x in ((.25,.25),(.5,.5),(.75,.75),(.3,.7),(.7,.3)):
                win=Window(max(0,int(ds.width*frac_x)-64),max(0,int(ds.height*frac_y)-64),
                           min(128,ds.width),min(128,ds.height))
                samples.append(ds.read(1,window=win))
            z=np.concatenate([a.reshape(-1) for a in samples])
            SOURCE_TABLE.append(dict(site=site,mosaic=name,crs=str(ds.crs),
                width=ds.width,height=ds.height,res_x=ds.res[0],res_y=ds.res[1],
                dtype=ds.dtypes[0],nodata=ds.nodata,
                sample_min=float(np.min(z)),sample_p99=float(np.percentile(z,99)),
                sample_max=float(np.max(z)),file=str(path)))
display(pd.DataFrame(SOURCE_TABLE).drop(columns='file'))
pd.DataFrame(SOURCE_TABLE).to_csv(OUT/'06_uav_teacher_audit.csv',index=False)
print('Five original M2 warmstart probability maps and orthomosaics located. No labels modified.')


## 07 · RAM-safe validation against NB04B

**The previous two implementations could trigger large memory use when resampling multi-gigapixel UAV probability rasters, even with GDAL VRT.** We now avoid original full-resolution rasters entirely. NB04 already produced **10, 30, 50, 75, 100 and 125 m cached UAV aggregates** on exactly the same satellite lattice. The cache stores continuous **mean Musanga probability (band 1)**, thresholded cover (band 2), and valid UAV coverage (band 3), all normalized to [0,1].

This section reads only the cached **100 m** files and compares them with known NB04B sample fractions; the existing **50 m** cache will be used downstream. No uint16 scale guessing or expensive native-raster reprojection. Rerun Sections 00–02, 06 and 07 after reconnecting to the new notebook.


In [ ]:
# 07 — FAST: reuse verified NB04 cached 10/50/100 m UAV aggregates.
# NO source UAV raster opened. NO WarpedVRT. NO full-resolution resampling.
from pathlib import Path
import rasterio,numpy as np,pandas as pd,math
from affine import Affine
from IPython.display import display

AGG_DIR=FINAL/'Outputs/V3_1/04_satellite_full_warmstart/cache/aggregates'
assert AGG_DIR.is_dir(),AGG_DIR
REFERENCE_DIAGNOSTICS=[]
UAV_SCALE={name:1.0 for names in UAV_MOSAICS.values() for name in names}

def satellite_grid(site,scale):
    path=NB04/'RAW_ANNUAL'/f'{site}_RAW_ANNUAL_{EMBEDDING_YEAR[site]}_10m.tif'
    with rasterio.open(path) as ds:
        assert ds.crs and abs(ds.res[0]-10)<.05
        return dict(crs=ds.crs,transform=ds.transform*Affine.scale(scale/10,scale/10),
                    width=int(math.ceil(ds.width*10/scale)),
                    height=int(math.ceil(ds.height*10/scale)))

def agg_file(name,scale):
    return AGG_DIR/f'{Path(name).stem}_{scale}m.tif'

def read_cached_teacher(name,site,scale):
    """Existing NB04 cache: band 1 mean probability, 2 cover>0.5, 3 UAV coverage.
    All three are ALREADY in [0,1], aligned to the canonical satellite grid.
    """
    path=agg_file(name,scale)
    assert path.is_file(),f'Cached {scale}m UAV aggregate missing: {path}'
    g=satellite_grid(site,scale)
    with rasterio.open(path) as src:
        assert src.count>=3,(name,scale,'unexpected bands',src.count)
        assert src.crs==g['crs'],(name,scale,'CRS mismatch')
        assert src.width==g['width'] and src.height==g['height'],(name,scale,'grid dimensions mismatch')
        assert src.transform.almost_equals(g['transform'],precision=1e-6),(name,scale,'grid alignment mismatch')
        a=src.read([1,3],out_dtype='float32')
    values=a[0];cov=a[1]
    return values,cov,g

for site,names in UAV_MOSAICS.items():
    for name in names:
        ref=DF.loc[(DF.site==site)&(DF.mosaic==name)]
        assert len(ref)>0,(site,name,'missing NB04B examples')
        values,cov,g=read_cached_teacher(name,site,100)
        rr=ref.row.to_numpy(dtype=int)
        cc=ref.col.to_numpy(dtype=int)
        valid_grid=(rr>=0)&(cc>=0)&(rr<g['height'])&(cc<g['width'])
        rr,cc=rr[valid_grid],cc[valid_grid]
        known=ref.mean_probability.to_numpy(dtype=float)[valid_grid]
        pred=values[rr,cc];coverage=cov[rr,cc]
        valid=(coverage>=0.90)&np.isfinite(pred)&np.isfinite(known)
        assert valid.sum()>=8,(name,'too few valid cells',int(valid.sum()))
        mae=float(np.mean(np.abs(pred[valid]-known[valid])))
        max_diff=float(np.max(np.abs(pred[valid]-known[valid])))
        REFERENCE_DIAGNOSTICS.append(dict(site=site,mosaic=name,n=int(valid.sum()),
            source='NB04_cached_100m',MAE_fraction=mae,max_abs_difference=max_diff,
            median_reference=float(np.median(known[valid])),
            median_cache=float(np.median(pred[valid]))))
        print('VERIFIED',site,Path(name).stem,'n',int(valid.sum()),'MAE',round(mae,7),
              'maximum',round(max_diff,7),flush=True)
        assert mae<0.001,(name,'NB04B cache mismatch',mae)
        del values,cov
CALIBRATION=pd.DataFrame(REFERENCE_DIAGNOSTICS)
display(CALIBRATION.round(7))
CALIBRATION.to_csv(OUT/'07_teacher_encoding_validation.csv',index=False)
print('Safe: all comparisons use small cached 100m rasters. No native UAV probability raster was read.')
print('Section 09 will use existing cached 50m probabilities; do not resample large UAV source files.')


## 08 · Low-memory visual alignment and teacher-distribution QA

The original teacher panels appeared **nearly uniformly pale yellow**, which is the `magma` colour for probability approximately 1. The previous preview did not use the aggregate raster's valid-UAV coverage band and could therefore display invalid/background values as real predictions.

This revised window-only preview reads the **10 m cached mean probability (band 1)** **and UAV coverage (band 3)** together. It masks coverage below 90%, shades invalid pixels gray, and prints within-window p05/p50/p95 and percentages near 0 or 1. It also prints the selected 100 m reference fraction for context. **Do not interpret or train on 10 m probability values if the diagnostics still show saturation at 1.** This remains display-only QA, not a full source-raster calibration.


In [ ]:
# 08 — Bounded window previews ONLY. No source-wide reproject/read.
from rasterio.windows import from_bounds as bounds_window
from rasterio.warp import transform_bounds
from rasterio.enums import Resampling

PANEL_M=350
PANEL_N=420

def preview_window(filepath,world_bounds,crs,bands,resampling=Resampling.bilinear):
    """Read one bounding-box window, decimated straight into a small array."""
    with rasterio.open(filepath) as ds:
        assert ds.crs and max(bands)<=ds.count,(filepath,bands)
        if ds.crs==crs:
            bb=world_bounds
        else:
            bb=transform_bounds(crs,ds.crs,*world_bounds,densify_pts=21)
        win=bounds_window(*bb,transform=ds.transform)
        # Protect against invalid geometry/projection or accidental huge windows.
        if win.width<=0 or win.height<=0:
            raise ValueError(f'Empty window: {filepath}')
        if win.width>1e7 or win.height>1e7:
            raise ValueError(f'Unreasonable source window: {filepath}, {win}')
        data=ds.read(bands,window=win,out_shape=(len(bands),PANEL_N,PANEL_N),
                     boundless=True,fill_value=0,resampling=resampling,
                     out_dtype='float32')
    return data

def show_rgb(cube):
    valid=np.isfinite(cube).all(axis=0)&(cube>0).all(axis=0)
    img=np.ones((*valid.shape,3),dtype='float32')
    if valid.sum()>5:
        lo=np.percentile(cube[:,valid],2,axis=1)
        hi=np.percentile(cube[:,valid],98,axis=1)
        stretch=(cube-lo[:,None,None])/(hi[:,None,None]-lo[:,None,None]+1e-6)
        img[valid]=np.clip(stretch.transpose(1,2,0)[valid],0,1)
    return img

fig,axes=plt.subplots(5,3,figsize=(13,20),layout='constrained')
prob_diagnostics=[]
idx=0
for site,names in UAV_MOSAICS.items():
    for name in names:
        ref=DF.loc[(DF.site==site)&(DF.mosaic==name)].sort_values('mean_probability')
        assert len(ref),name
        pick=ref.iloc[len(ref)//2]
        g=satellite_grid(site,100)
        cx,cy=g['transform']*(float(pick.col)+.5,float(pick.row)+.5)
        half=PANEL_M/2
        bounds=(cx-half,cy-half,cx+half,cy+half)
        # Native UAV RGB is window-read, never loaded as a complete mosaic.
        uav_rgb=preview_window(MOSAIC_PATH[name],bounds,g['crs'],[1,2,3])
        # Planet raw median RGB: R/G/B are bands 15,14,13.
        planet_rgb=preview_window(planet_file(site,'Annual'),bounds,g['crs'],[15,14,13])
        # Existing 10m NB04 UAV aggregate; band 1 is normalized mean probability.
        # NB04 aggregate bands: 1 mean probability, 2 fraction >0.5, 3 valid UAV coverage.
        # Read mask and target together: otherwise invalid/background pixels may
        # be rendered as high probability, creating a deceptively solid yellow map.
        uc=preview_window(agg_file(name,10),bounds,g['crs'],[1,3],
                          resampling=Resampling.nearest)
        prob,coverage=uc[0],uc[1]
        good=np.isfinite(prob)&np.isfinite(coverage)&(coverage>=0.90)&(prob>=0)&(prob<=1)
        if good.any():
            pv=prob[good]
            stats=dict(site=site,mosaic=name,n_valid=int(good.sum()),
                       window_valid_pct=100*float(good.mean()),
                       p05=float(np.percentile(pv,5)),p50=float(np.median(pv)),
                       p95=float(np.percentile(pv,95)),share_gt_09_pct=100*float(np.mean(pv>=.9)),
                       share_lt_01_pct=100*float(np.mean(pv<=.1)),
                       reference_100m_fraction=float(pick.mean_probability))
        else:
            stats=dict(site=site,mosaic=name,n_valid=0,window_valid_pct=0,
                       p05=np.nan,p50=np.nan,p95=np.nan,share_gt_09_pct=np.nan,
                       share_lt_01_pct=np.nan,reference_100m_fraction=float(pick.mean_probability))
        prob_diagnostics.append(stats)
        axes[idx,0].imshow(show_rgb(uav_rgb))
        axes[idx,1].imshow(show_rgb(planet_rgb))
        cmap=plt.get_cmap('magma').copy()
        cmap.set_bad('#dddddd')
        axes[idx,2].imshow(np.ma.masked_where(~good,prob),cmap=cmap,
                           vmin=0,vmax=1,interpolation='nearest')
        axes[idx,0].set_title(f'{site} {Path(name).stem}: UAV RGB')
        axes[idx,1].set_title(f'Planet {PLANET_YEAR[site]} raw median RGB')
        axes[idx,2].set_title('UAV fraction (10m, valid UAV pixels only)')
        for ax in axes[idx]: ax.axis('off')
        idx+=1
        print('PREVIEW',site,Path(name).stem,'finished',flush=True)
DIAG08=pd.DataFrame(prob_diagnostics)
display(DIAG08.round(3))
DIAG08.to_csv(OUT/'08_teacher_probability_diagnostics.csv',index=False)
print('WARNING if a 10m map has >90% of valid pixels near 1: possible masking,')
print('scaling or band-semantic issue; investigate before training a pixel U-Net.')
fig.savefig(FIG/'08_uav_planet_teacher_alignment.png',dpi=135,bbox_inches='tight')
plt.show()
plt.close(fig)
print('Window-only QA complete: examine broad landscape alignment, not individual crowns.')


## 09 · Build the 50 m training-target table from **existing verified NB04 cached aggregates**

Avoid rerunning expensive UAV raster aggregation. NB04 already stored canonical 50 m continuous Musanga fractions and valid-pixel fractions for each original mosaic. Section 07 checks the 100 m caches against NB04B samples; this section reads the corresponding 50 m caches, intersects them with `ROI.shp`, and keeps cells with at least 90% valid UAV coverage and 90% ROI membership. Source cache is not modified. This is independent of Planet availability; Section 10 checks its patches.


In [ ]:
# 09 — Build continuous 50m UAV labels (per-mosaic), enforce NB04B-style ROI/coverage rules.
import geopandas as gpd
from rasterio.features import rasterize
ROI_SOURCE=Path(uav_manifest['roi'])
assert ROI_SOURCE.is_file()
ROI_GDF=gpd.read_file(ROI_SOURCE)
assert ROI_GDF.crs and not ROI_GDF.empty
TARGET_SCALE=50
MIN_UAV_COVERAGE=.90
MIN_ROI_FRAC=.90
SUPERSAMPLE=4
TARGETS=[]
for site,names in UAV_MOSAICS.items():
    g=satellite_grid(site,TARGET_SCALE)
    polys=ROI_GDF.to_crs(g['crs'])
    fine_transform=g['transform']*Affine.scale(1/SUPERSAMPLE,1/SUPERSAMPLE)
    fine=rasterize(((geom,1) for geom in polys.geometry if geom is not None and not geom.is_empty),
        out_shape=(g['height']*SUPERSAMPLE,g['width']*SUPERSAMPLE),
        transform=fine_transform,fill=0,dtype='uint8')
    roif=fine.reshape(g['height'],SUPERSAMPLE,g['width'],SUPERSAMPLE).mean(axis=(1,3))
    del fine
    for name in names:
        probs,coverage,_=read_cached_teacher(name,site,TARGET_SCALE)
        ok=(coverage>=MIN_UAV_COVERAGE)&(roif>=MIN_ROI_FRAC)&np.isfinite(probs)&(probs>=0)&(probs<=1)
        yy,xx=np.where(ok)
        for row,col in zip(yy,xx):
            x,y=g['transform']*(int(col)+.5,int(row)+.5)
            TARGETS.append(dict(site=site,mosaic=name,group=Path(name).stem,
                year_uav=EMBEDDING_YEAR[site],year_planet=PLANET_YEAR[site],
                scale_m=TARGET_SCALE,row=int(row),col=int(col),
                x=float(x),y=float(y),crs=str(g['crs']),
                mean_probability=float(probs[row,col]),
                uav_coverage=float(coverage[row,col]),
                roi_fraction=float(roif[row,col])))
        print('50m supervised cells',site,name,len(yy),flush=True)
assert TARGETS, 'No 50m cells: inspect calibration / ROI / CRS'
T50=pd.DataFrame(TARGETS)
assert not T50.duplicated(['site','mosaic','row','col']).any()
T50.to_parquet(OUT/'09_uav_targets_50m.parquet',index=False)
display(T50.groupby(['site','group']).agg(cells=('mean_probability','size'),
    mean_fraction=('mean_probability','mean'),
    max_fraction=('mean_probability','max'),
    min_uav_coverage=('uav_coverage','min')).round(3))
print('Saved verified 50m UAV-derived labels:',OUT/'09_uav_targets_50m.parquet')


## 10 · Full 50 m spatial-feature extraction — restartable and RAM-aware

Runs on **all 14,454 target cells** from Section 09 (not just 300 per site). Reads **native 4.77 m** Planet reflectance and NIR texture neighborhoods and **10 m** embedding neighborhoods without creating coarsened satellite rasters. The target remains the existing continuous UAV-derived Musanga fraction on the 50 m grid.

**Features**: Planet B/G/R/N patch means + SD (21×21 Planet pixels, ~100 m), medians of four existing Planet NIR texture layers, and **all 64** embedding channel values at the patch centre, context means and SDs (11×11 embedding pixels, ~110 m). The wider neighborhood supplies spatial context; it does **not** change the 50 m response support.

**Restart behaviour**: deterministic 200-cell chunks, written as Parquet with an entry for **every** input cell, including rejected patches and reason. Completed chunks are checked and skipped on rerun. Each run uses an input/configuration fingerprint (target coordinates, labels, source file metadata and window settings); changes create a new cache namespace, preventing stale reuse. A consolidated feature table includes only accepted rows, and a companion QA file records rejected cells. Source imagery and previous 300-cell pilot are preserved.

**Memory/I/O**: only small raster windows are read. Opens each site's three sources once; no full mosaics and no 64-channel cubes across the whole forest. Outputs progress after each chunk. The larger MOS dataset is extracted incrementally.

You can safely interrupt after any saved batch. Reopen Colab and rerun initialization, Section 09 (or load its existing Parquet file), then Section 10. No CNN/model training occurs here.


In [ ]:
# 10 — Full, restartable native Planet + embedding spatial feature extraction.
# Requires setup cells (FINAL, OUT, NB04...), Section 01 (planet_file/embed_file),
# and Section 09 (T50) or the existing Section 09 Parquet target file.
from pathlib import Path
import hashlib,json,os,gc,time
import numpy as np,pandas as pd,rasterio
from rasterio.windows import Window
from pyproj import Transformer
from IPython.display import display

TARGET_FILE=OUT/'09_uav_targets_50m.parquet'
if 'T50' not in globals():
    assert TARGET_FILE.is_file(),f'Run Section 09 first: {TARGET_FILE}'
    T50=pd.read_parquet(TARGET_FILE)
assert len(T50)>0 and {'site','mosaic','row','col','x','y','crs','mean_probability'}.issubset(T50.columns)
BATCH_CELLS=200
PLANET_PATCH=21  # 21 x 4.77m ~100m, centred on 50m response cell
EMB_PATCH=11     # 11 x 10m =110m embedding context
MIN_PATCH_VALID=.80
MAX_NEW_BATCHES=None  # Set e.g. 3 for a short smoke test; None = all outstanding.
FORCE_NEW_EXTRACTION=False  # True adds a new cache version; normally leave False.
EXTRACTION_VERSION='v2_native_planet_embed_20261024'
COLS_KEY=['site','mosaic','row','col']
T50=T50.sort_values(COLS_KEY,kind='stable').reset_index(drop=True)
assert not T50.duplicated(COLS_KEY).any(),'Duplicated input target cell keys'

sources={}
for site in SITES:
    sources[site]=dict(annual=planet_file(site,'Annual'),
                       texture=planet_file(site,'Texture'),
                       embeddings=embed_file(site))
    for path in sources[site].values():assert path.is_file(),path

# Source identity (path + size + modification nanoseconds) and every target
# value/position must match for a chunk to be reusable.
config=dict(version=EXTRACTION_VERSION,planet_patch=PLANET_PATCH,
            embedding_patch=EMB_PATCH,min_patch_valid=MIN_PATCH_VALID,
            chunk_size=BATCH_CELLS,
            source_identity={s:{k:dict(path=str(v),size=v.stat().st_size,
                mtime_ns=v.stat().st_mtime_ns) for k,v in paths.items()}
                for s,paths in sources.items()},
            targets_sha256=hashlib.sha256(
                pd.util.hash_pandas_object(
                    T50[['site','mosaic','row','col','x','y','crs','mean_probability']],
                    index=False).values.tobytes()).hexdigest())
config_json=json.dumps(config,sort_keys=True)
run_id=hashlib.sha256(config_json.encode()).hexdigest()[:16]
if FORCE_NEW_EXTRACTION:
    # Manual new run without changing source files
    EXTRACTION_VERSION+='__manual'
    config['version']=EXTRACTION_VERSION
    run_id=hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()[:16]
CACHE10=OUT/'10_full_spatial_features_chunks'/run_id
CACHE10.mkdir(parents=True,exist_ok=True)
manifest_path=CACHE10/'config.json'
if manifest_path.exists():
    assert json.loads(manifest_path.read_text())==config,'Cached run settings changed unexpectedly'
else:
    manifest_path.write_text(json.dumps(config,indent=2,sort_keys=True))

def one_patch(ds,bands,size,x,y,source_crs,transformer):
    tx,ty=transformer.transform(x,y)
    px,py=(~ds.transform)*(tx,ty)
    # Use a deterministic centre aligned to nearest native pixel.
    c=int(np.floor(px));r=int(np.floor(py))
    h=size//2
    return ds.read(bands,window=Window(c-h,r-h,size,size),
                   boundless=True,fill_value=0,out_dtype='float32')

def extract_batch(site,block):
    out=[]
    with rasterio.open(sources[site]['annual']) as pds,\
         rasterio.open(sources[site]['texture']) as tds,\
         rasterio.open(sources[site]['embeddings']) as eds:
        trans={id(ds):Transformer.from_crs(block.crs.iloc[0],ds.crs,always_xy=True)
               for ds in (pds,tds,eds)}
        for rec in block.itertuples(index=False):
            base=dict(site=site,mosaic=rec.mosaic,group=rec.group,
                      row=int(rec.row),col=int(rec.col),
                      target_fraction=float(rec.mean_probability),
                      x=float(rec.x),y=float(rec.y),crs=str(rec.crs))
            try:
                if str(rec.crs)!=str(block.crs.iloc[0]):
                    raise ValueError('MIXED_CRS_IN_BATCH')
                p=one_patch(pds,[13,14,15,16],PLANET_PATCH,rec.x,rec.y,
                            rec.crs,trans[id(pds)])
                t=one_patch(tds,list(range(1,10)),PLANET_PATCH,rec.x,rec.y,
                            rec.crs,trans[id(tds)])
                e=one_patch(eds,list(range(1,65)),EMB_PATCH,rec.x,rec.y,
                            rec.crs,trans[id(eds)])
                pm=np.isfinite(p).all(axis=0)&(p>0).all(axis=0)
                em=np.isfinite(e).all(axis=0)&np.any(e!=0,axis=0)
                tv=np.isfinite(t).all(axis=0)&(t>=0).all(axis=0)
                base.update(planet_valid=float(pm.mean()),
                            embedding_valid=float(em.mean()),
                            planet_texture_valid=float(tv.mean()))
                if pm.mean()<MIN_PATCH_VALID:
                    base.update(accepted=False,reason='PLANET_LOW_COVERAGE')
                elif em.mean()<MIN_PATCH_VALID:
                    base.update(accepted=False,reason='EMBEDDING_LOW_COVERAGE')
                elif tv.mean()<MIN_PATCH_VALID:
                    base.update(accepted=False,reason='TEXTURE_LOW_COVERAGE')
                else:
                    px=p[:,pm]
                    ex=e[:,em]
                    center=e[:,EMB_PATCH//2,EMB_PATCH//2]
                    if not np.isfinite(center).all():
                        base.update(accepted=False,reason='EMBEDDING_CENTRE_INVALID')
                    else:
                        base.update(accepted=True,reason='OK')
                        for i,band in enumerate(['B','G','R','N']):
                            base[f'PLANET_mean_{band}']=float(px[i].mean())
                            base[f'PLANET_std_{band}']=float(px[i].std())
                        for i,band in enumerate(['NIR_CV_r1','NIR_CV_r3',
                                                  'NIR_CV_r7','NIR_highpass']):
                            v=t[i,tv]
                            base[f'PLANET_texture_{band}']=float(np.median(v))
                        means=ex.mean(axis=1);stds=ex.std(axis=1)
                        for j in range(64):
                            base[f'EMB_center_{j+1:02d}']=float(center[j])
                            base[f'EMB_mean_{j+1:02d}']=float(means[j])
                            base[f'EMB_sd_{j+1:02d}']=float(stds[j])
            except (OSError,ValueError,rasterio.errors.RasterioIOError) as exc:
                base.update(accepted=False,reason='READ_ERROR',error=str(exc)[:250])
            out.append(base)
    return pd.DataFrame(out)

# Separate every accepted/rejected target record; one output row per input cell.
# On a crash, only the currently active unsaved chunk needs recomputing.
work=[]
for site in SITES:
    site_cells=T50.loc[T50.site==site].reset_index(drop=True)
    for offset in range(0,len(site_cells),BATCH_CELLS):
        block=site_cells.iloc[offset:offset+BATCH_CELLS].copy()
        key=f'{site}_{offset:06d}_{len(block):03d}'
        work.append((site,block,CACHE10/f'{key}.parquet'))
print('FULL DATASET:',len(T50),'cells;',len(work),'restartable chunks;',
      'cache',CACHE10,flush=True)
print('Existing 300/site pilot is untouched.',flush=True)

new_batches=0
start=time.monotonic()
for k,(site,block,path) in enumerate(work,1):
    if path.exists():
        try:
            old=pd.read_parquet(path,columns=COLS_KEY+['accepted','reason'])
            expect=block[COLS_KEY].reset_index(drop=True)
            if len(old)==len(block) and old[COLS_KEY].reset_index(drop=True).equals(expect):
                print(f'[{k}/{len(work)}] CACHED {site} {path.name}: '
                      f'{int(old.accepted.sum())}/{len(old)} accepted',flush=True)
                continue
        except Exception as exc:
            print('INCOMPLETE CACHE ignored:',path.name,repr(exc),flush=True)
        # Do not trust a partial or stale cache; regenerate this block.
    if MAX_NEW_BATCHES is not None and new_batches>=MAX_NEW_BATCHES:
        print('Reached MAX_NEW_BATCHES; rerun to continue.',flush=True)
        break
    t0=time.monotonic()
    data=extract_batch(site,block)
    assert len(data)==len(block) and data[COLS_KEY].reset_index(drop=True).equals(
        block[COLS_KEY].reset_index(drop=True)),('Unexpected extraction row mismatch',site,k)
    tmp=path.with_suffix('.parquet.partial')
    data.to_parquet(tmp,index=False)
    os.replace(tmp,path)
    new_batches+=1
    print(f'[{k}/{len(work)}] SAVED {site} {path.name}: '
          f'{int(data.accepted.sum())}/{len(data)} accepted '
          f'({time.monotonic()-t0:.1f}s; overall {time.monotonic()-start:.1f}s)',
          flush=True)
    del data
    gc.collect()

# Only consolidate if every chunk is present (otherwise keep reusable batches).
missing=[str(path) for _,_,path in work if not path.exists()]
if missing:
    print('EXTRACTION PARTIAL:',len(missing),'remaining chunks. Rerun Section 10 to resume.')
else:
    # Bounded accumulated table: 14,454 rows × ~220 columns, modest in memory.
    all_rows=pd.concat([pd.read_parquet(path) for _,_,path in work],ignore_index=True)
    assert len(all_rows)==len(T50)
    assert all_rows[COLS_KEY].equals(T50[COLS_KEY]),'Joined rows out of order'
    accepted=all_rows.loc[all_rows.accepted].copy()
    rejected=all_rows.loc[~all_rows.accepted].copy()
    full_file=OUT/'10_full_spatial_features_50m.parquet'
    qa_file=OUT/'10_full_spatial_features_rejections_50m.parquet'
    report=OUT/'10_full_spatial_features_site_report.csv'
    accepted.to_parquet(full_file,index=False)
    rejected.to_parquet(qa_file,index=False)
    report_df=all_rows.groupby('site').agg(
        total=('accepted','size'),accepted=('accepted','sum'),
        mean_planet_valid=('planet_valid','mean'),
        mean_embedding_valid=('embedding_valid','mean')).reset_index()
    report_df['rejected']=report_df.total-report_df.accepted
    display(report_df.round(3))
    print('Rejection reasons:')
    display(all_rows.groupby(['site','reason']).size().rename('cells').reset_index())
    report_df.to_csv(report,index=False)
    print('COMPLETE:',len(accepted),'accepted /',len(all_rows),'targets.')
    print('Features:',full_file)
    print('QA rejected:',qa_file)
    print('Summary:',report)
    print('No model fit; preserve geographic groups for spatial holdout.')


## 11 · Validated spatial-feature ablation against S2 + embeddings (50 m)

**Objective:** quantify the *incremental* predictive value of 4.77 m Planet spectral/texture summaries and 10 m embedding spatial context beyond an identically constructed **50 m S2 annual + three-year quarterly + annual embedding** baseline.

**Feature sets (nested; same target cells and evaluation folds):**
- **B0:** 50 m S2 annual + 3-year quarterly + 50 m area-averaged 64-band annual embeddings.
- **B1:** B0 + Planet 4-band patch means/SD.
- **B2:** B0 + four Planet NIR texture statistics.
- **B3:** B0 + embedding neighborhood means/SD.
- **B4:** B0 + all Planet spectral and texture features.
- **B5:** B0 + all Planet + embedding spatial features.

**Important methodological distinction:** Unlike the earlier NB04B pilot, this block constructs predictors at **50 m** and uses **all 14,454 labelled cells** as candidate test observations. It aggregates baseline 10 m predictors onto the 50 m lattice (annual and quarterly S2, annual embeddings), while Planet statistics use native 4.77 m neighborhoods. The embedding spatial features use 10 m neighborhoods. These are different predictor supports; the CNN experiments will subsequently operate on whole image patches.

**Validation:** (a) 3 deterministic ~400 m held-out spatial-block folds, with a **150 m train–test exclusion buffer** within forests; (b) strict 4-forest leave-one-site-out (MOS 1–2 and 5–6 always held out together). Evaluation cells are fixed across models within each scheme, and training is **forest-balanced** (up to 1,500 cells per forest per fold, after excluding test/buffer). Spatially adjacent windows and MOS overlaps never cross the split boundary. Predicting on all eligible held-out cells still preserves the true imbalanced site distribution; report *equal-forest macro scores* alongside pooled metrics.

**Algorithms:** Histogram Gradient Boosting (HGB), Random Forest (RF) and Extra Trees (ET) are fit on identical folds, training-cell selections and six predictor sets. All **42 existing HGB prediction checkpoints remain unchanged and reusable**; the RF and ET variants are saved with algorithm suffixes, for 126 model/fold evaluations in total. RF/ET use 150 trees, minimum leaf size 5, 75% predictors considered at a split and 2 threads. Compare B0→B1–B5 within each algorithm, not simply the overall winning score.\n\n**Restartability and resource use:** Baseline 50 m inputs are built site-by-site from the existing 10 m GeoTIFFs and cached using source/target fingerprints. Each model/fold has a checkpoint of the **held-out predictions**, written atomically. No source UAV mosaics are opened. RF-like boosting uses one CPU thread and a fixed iteration budget to fit reasonably on ordinary Colab RAM. Use `MAX_NEW_FITS=3` for a smoke test, then `None` for completion.

**Outputs:** fold prediction Parquets, fold metrics, equal-forest and pooled summaries, per-site held-out metrics, MOS high-fraction bias, and scatterplots from held-out predictions. **R² can be negative** on strict site transfer; report it without suppressing failures. This is an assessment of UAV-derived pseudo-labels, not independently measured Musanga presence.


In [ ]:
# 11A — Assemble exact same 50m cells: existing spatial features + 50m S2/embeddings.
import os,json,hashlib,math,time,gc
from pathlib import Path
import numpy as np,pandas as pd,rasterio
from rasterio.warp import reproject,Resampling
from affine import Affine
from IPython.display import display

BLOCK11=OUT/'11_spatial_ablation_50m'
INPUT11=BLOCK11/'inputs'
FOLD11=BLOCK11/'fold_predictions'
FIG11=BLOCK11/'figures'
for folder in (BLOCK11,INPUT11,FOLD11,FIG11):
    folder.mkdir(parents=True,exist_ok=True)
DATA11=OUT/'10_full_spatial_features_50m.parquet'
assert DATA11.is_file(),'Complete Section 10 first.'
SET11={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2020}
QWINDOW11={'Baego':(2021,2023),'MOS':(2022,2024),
           'Yangambi':(2022,2024),'Yoko':(2018,2020)}
COLUMN_KEY=['site','mosaic','row','col']
SPATIAL_FEATURES11=pd.read_parquet(DATA11)
assert SPATIAL_FEATURES11.accepted.all(),'Unexpected rejected examples in accepted-only dataset'
assert not SPATIAL_FEATURES11.duplicated(COLUMN_KEY).any()
assert len(SPATIAL_FEATURES11)==14454,('Unexpected number of training cells',len(SPATIAL_FEATURES11))
SPATIAL_FEATURES11=SPATIAL_FEATURES11.sort_values(COLUMN_KEY,kind='stable').reset_index(drop=True)
S2_SOURCE11={}
for site in SITES:
    yr=SET11[site];a,b=QWINDOW11[site]
    S2_SOURCE11[site]={
        'RAW_ANNUAL':NB04/'RAW_ANNUAL'/f'{site}_RAW_ANNUAL_{yr}_10m.tif',
        'S2_QUARTERLY_3YR':NB04/'S2_QUARTERLY_3YR'/f'{site}_S2_QUARTERLY_3YR_{a}_{b}_10m.tif',
        'EMB_ANNUAL':NB04/'EMB_ANNUAL'/f'{site}_EMB_ANNUAL_{yr}_10m.tif'}
    for family,path in S2_SOURCE11[site].items():
        assert path.is_file(),('Missing 10m predictor',site,family,path)
INPUT_FINGERPRINT11=hashlib.sha256(json.dumps({
    'version':'B11_50m_area_averaged_20261024_v1',
    'feature_file':dict(path=str(DATA11),size=DATA11.stat().st_size,
                        mtime=DATA11.stat().st_mtime_ns),
    'sources':{s:{k:(str(p),p.stat().st_size,p.stat().st_mtime_ns)
                   for k,p in fam.items()} for s,fam in S2_SOURCE11.items()},
},sort_keys=True).encode()).hexdigest()[:16]
DATA_CACHE11=INPUT11/f'combined_50m_{INPUT_FINGERPRINT11}.parquet'

def mean_baseline_site(site,local):
    # Reproject/average each satellite band from 10m directly onto native 50m
    # reference lattice. Destination is only ~1/25 original pixel count.
    with rasterio.open(S2_SOURCE11[site]['RAW_ANNUAL']) as ref:
        assert ref.crs and abs(ref.res[0]-10)<.05
        tfm=ref.transform*Affine.scale(5,5)
        width=int(math.ceil(ref.width/5));height=int(math.ceil(ref.height/5))
        crs=ref.crs
    rr=local.row.to_numpy(dtype=int);cc=local.col.to_numpy(dtype=int)
    assert ((rr>=0)&(rr<height)&(cc>=0)&(cc<width)).all(),(site,'cell outside canonical grid')
    arrs={}
    for family,filepath in S2_SOURCE11[site].items():
        with rasterio.open(filepath) as src:
            assert src.crs==crs,(site,family,'CRS differs')
            expected=44 if family=='S2_QUARTERLY_3YR' else (64 if family=='EMB_ANNUAL' else None)
            if expected is not None:assert src.count==expected,(site,family,src.count)
            for band in range(1,src.count+1):
                dest=np.full((height,width),np.nan,dtype='float32')
                # Quarterly every 11th band = clear-observation count: nearest;
                # remaining reflectance channels use area-weighted averaging.
                res=Resampling.nearest if family=='S2_QUARTERLY_3YR' and band%11==0 else Resampling.average
                reproject(rasterio.band(src,band),dest,
                          dst_transform=tfm,dst_crs=crs,
                          src_nodata=src.nodata,dst_nodata=np.nan,
                          resampling=res,num_threads=1)
                arrs[f'{family}__{band:02d}']=dest[rr,cc].copy()
                del dest
        print('BASELINE',site,family,'bands',src.count,flush=True)
    return pd.DataFrame(arrs,index=local.index)

if DATA_CACHE11.exists():
    D11=pd.read_parquet(DATA_CACHE11)
    assert len(D11)==len(SPATIAL_FEATURES11)
    assert D11[COLUMN_KEY].equals(SPATIAL_FEATURES11[COLUMN_KEY])
    print('CACHED 50m baseline + native spatial features:',DATA_CACHE11,flush=True)
else:
    assembled=[]
    for site in SITES:
        local=SPATIAL_FEATURES11.loc[SPATIAL_FEATURES11.site==site].copy()
        features=mean_baseline_site(site,local)
        joined=pd.concat([local,features],axis=1)
        part=INPUT11/f'{site}_{INPUT_FINGERPRINT11}.parquet'
        joined.to_parquet(part,index=False)
        assembled.append(joined)
        print('SAVED site input:',site,len(joined),flush=True)
        del local,features,joined
        gc.collect()
    D11=pd.concat(assembled,ignore_index=True).sort_values(COLUMN_KEY,kind='stable').reset_index(drop=True)
    D11.to_parquet(DATA_CACHE11,index=False)

BASE11=[x for x in D11.columns if x.startswith(('RAW_ANNUAL__','S2_QUARTERLY_3YR__','EMB_ANNUAL__'))]
SPEC11=[f'PLANET_{k}_{b}' for k in ('mean','std') for b in ('B','G','R','N')]
TEX11=[f'PLANET_texture_{b}' for b in ('NIR_CV_r1','NIR_CV_r3','NIR_CV_r7','NIR_highpass')]
EMBCTX11=[f'EMB_{stat}_{j:02d}' for stat in ('mean','sd') for j in range(1,65)]
FEATURE_SETS11={
    'B0_S2_EMB':BASE11,
    'B1_B0_PlanetSpectral':BASE11+SPEC11,
    'B2_B0_PlanetTexture':BASE11+TEX11,
    'B3_B0_EmbeddingSpatial':BASE11+EMBCTX11,
    'B4_B0_AllPlanet':BASE11+SPEC11+TEX11,
    'B5_B0_AllSpatial':BASE11+SPEC11+TEX11+EMBCTX11,
}
assert len(BASE11)>=108,(len(BASE11),'baseline should include 64 embeddings + 44 quarterly + annual S2')
all_columns=sorted(set(sum(FEATURE_SETS11.values(),[])))
missing=[x for x in all_columns if x not in D11.columns]
assert not missing,missing
# Exactly the SAME finite and valid target cells for all feature sets.
clearcols=[f'S2_QUARTERLY_3YR__{band:02d}' for band in (11,22,33,44)]
assert all(x in D11.columns for x in clearcols)
valid=np.isfinite(D11[all_columns].to_numpy(dtype='float32')).all(axis=1)
valid &= (D11[clearcols].to_numpy(dtype='float32')>=1).all(axis=1)
valid &= D11.target_fraction.between(0,1).to_numpy()
display(pd.DataFrame([dict(total=len(D11),common_complete=int(valid.sum()),
    rejected=int((~valid).sum()),features_B0=len(BASE11),
    features_B5=len(FEATURE_SETS11['B5_B0_AllSpatial']))]))
D11=D11.loc[valid].reset_index(drop=True)
assert D11.site.nunique()==4 and len(D11)>1000,'Too few eligible cells; inspect quarterly coverage.'
display(D11.groupby('site').target_fraction.agg(['size','mean','max']).round(3))
print('50m common-case dataset:',len(D11),'rows; source:',DATA_CACHE11)


In [ ]:
# 11B — Restartable spatial and forest-held-out benchmarking + reporting.
import os,json,hashlib,math,time
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from sklearn.ensemble import HistGradientBoostingRegressor,RandomForestRegressor,ExtraTreesRegressor
from sklearn.metrics import r2_score
from IPython.display import display

SPATIAL_BLOCK_M=400
BUFFER_M=150
SPATIAL_FOLDS=3
TRAIN_CAP_PER_FOREST=1500
MODEL_SEED=20261024
MAX_NEW_FITS=None  # Use 3 for a short smoke test; None completes remaining RF/ET jobs.
ALGORITHMS11=['HGB','RF','ET']  # HGB checkpoint names remain unchanged.
RF_ET_ARGS=dict(n_estimators=150,min_samples_leaf=5,max_features=0.75,
                max_depth=None,n_jobs=2,random_state=MODEL_SEED)
def estimator11(algorithm):
    if algorithm=='HGB':return HistGradientBoostingRegressor(**MODEL_ARGS)
    if algorithm=='RF':return RandomForestRegressor(**RF_ET_ARGS)
    if algorithm=='ET':return ExtraTreesRegressor(**RF_ET_ARGS)
    raise ValueError(algorithm)
HIGH_COVER=.40
MODEL_ARGS=dict(learning_rate=.06,max_iter=180,max_leaf_nodes=20,
                min_samples_leaf=35,l2_regularization=2,
                early_stopping=False,random_state=MODEL_SEED)
# Keep memory/time consistent on Colab; models are serial.
os.environ['OMP_NUM_THREADS']='1'
import threadpoolctl
threadpoolctl.threadpool_limits(limits=1)

D11=D11.reset_index(drop=True).copy()
# Projected x/y from Section 09, units metres, each site in EPSG:32635.
assert D11.crs.nunique()==1 and '32635' in str(D11.crs.iloc[0])
xx=D11.x.to_numpy(float);yy=D11.y.to_numpy(float)
assert np.isfinite(xx).all() and np.isfinite(yy).all()
# Stable coordinates; label nearby 400m tiles across all sites.
bx=np.floor(xx/SPATIAL_BLOCK_M).astype('int64')
by=np.floor(yy/SPATIAL_BLOCK_M).astype('int64')
# Deterministic 3-colour spatial blocks (not random 50m cell splitting).
blockid=(np.mod(bx*73856093+by*19349663,SPATIAL_FOLDS)).astype('int8')
D11['spatial_fold']=blockid
schemes11=['BLOCK_BUFFERED','LEAVE_FOREST_OUT']
folds11=[]
for scheme in schemes11:
    if scheme=='BLOCK_BUFFERED':
        folds11.extend((scheme,str(i)) for i in range(SPATIAL_FOLDS))
    else:
        folds11.extend((scheme,site) for site in SITES)

def split11(scheme,heldout):
    test=(D11.spatial_fold.to_numpy()==int(heldout)) if scheme=='BLOCK_BUFFERED' else (
        D11.site.to_numpy()==heldout)
    train=~test
    if scheme=='BLOCK_BUFFERED':
        # Strict within-forest 150m exclusion around EVERY heldout centre.
        sites=D11.site.to_numpy()
        for site in SITES:
            tix=np.where(test&(sites==site))[0]
            ix=np.where(train&(sites==site))[0]
            if not len(tix) or not len(ix):continue
            tree=cKDTree(np.column_stack([xx[tix],yy[tix]]))
            near=tree.query(np.column_stack([xx[ix],yy[ix]]),k=1)[0] < BUFFER_M
            train[ix[near]]=False
    assert not (train & test).any()
    # Forest-balanced fit; retaining every heldout test cell for metrics.
    rng=np.random.default_rng(MODEL_SEED)
    train_ix=[]
    for site in SITES:
        eligible=np.where(train & (D11.site.to_numpy()==site))[0]
        if len(eligible)>TRAIN_CAP_PER_FOREST:
            eligible=rng.choice(eligible,TRAIN_CAP_PER_FOREST,replace=False)
        train_ix.extend(eligible)
    train_ix=np.asarray(sorted(train_ix),dtype=int)
    test_ix=np.where(test)[0]
    assert len(train_ix)>=100 and len(test_ix)>=10,(scheme,heldout,len(train_ix),len(test_ix))
    return train_ix,test_ix,int((~train&~test).sum())

def metric11(y,p):
    y=np.asarray(y,float);p=np.asarray(p,float)
    err=p-y
    high=y>=HIGH_COVER
    q25,q75=np.percentile(y,[25,75])
    return dict(n=len(y),R2=float(r2_score(y,p)) if len(y)>1 and np.var(y)>1e-12 else np.nan,
        MAE_pp=float(100*np.mean(abs(err))),RMSE_pp=float(100*np.sqrt(np.mean(err**2))),
        bias_pp=float(100*np.mean(err)),
        RPD=float(np.std(y)/np.sqrt(np.mean(err**2))) if np.mean(err**2)>0 else np.nan,
        RPIQ=float((q75-q25)/np.sqrt(np.mean(err**2))) if np.mean(err**2)>0 else np.nan,
        n_high=int(high.sum()),high_bias_pp=float(100*np.mean(err[high])) if high.any() else np.nan,
        high_recall=float(np.mean(p[high]>=HIGH_COVER)) if high.any() else np.nan)

signature=hashlib.sha256(json.dumps(dict(
    input_fingerprint=INPUT_FINGERPRINT11,feature_sets=FEATURE_SETS11,
    complete_case_keys=hashlib.sha256(pd.util.hash_pandas_object(
        D11[COLUMN_KEY],index=False).values.tobytes()).hexdigest(),
    block_m=SPATIAL_BLOCK_M,buffer_m=BUFFER_M,spatial_folds=SPATIAL_FOLDS,
    cap=TRAIN_CAP_PER_FOREST,seed=MODEL_SEED,model=MODEL_ARGS),sort_keys=True).encode()).hexdigest()[:16]
RUN11=FOLD11/signature
RUN11.mkdir(parents=True,exist_ok=True)
(RUN11/'config.json').write_text(json.dumps(dict(signature=signature,model=MODEL_ARGS,
    spatial_block_m=SPATIAL_BLOCK_M,buffer_m=BUFFER_M),indent=2))
print('MODEL RUN:',signature,'folds:',folds11,'features:',list(FEATURE_SETS11),'algorithms:',ALGORITHMS11)
print('Restartable fold prediction cache:',RUN11,flush=True)
completed=0;new_fits=0;fold_report=[]
for scheme,heldout in folds11:
    train_ix,test_ix,n_excluded=split11(scheme,heldout)
    print('FOLD',scheme,heldout,'training',len(train_ix),'testing',len(test_ix),
          'buffer excluded',n_excluded,flush=True)
    fold_report.append(dict(scheme=scheme,heldout=heldout,n_train=len(train_ix),
                            n_test=len(test_ix),n_buffer_excluded=n_excluded))
    for algorithm in ALGORITHMS11:
        for set_name,cols in FEATURE_SETS11.items():
            # HGB filenames are deliberately unchanged to reuse all 42 prior fits.
            stem=f'{scheme}_{heldout}_{set_name}'
            if algorithm!='HGB': stem+=f'_{algorithm}'
            path=RUN11/f'{stem}.parquet'
            if path.exists():
                try:
                    head=pd.read_parquet(path,columns=COLUMN_KEY+['observed','predicted'])
                    expect=D11.iloc[test_ix][COLUMN_KEY].reset_index(drop=True)
                    if len(head)==len(test_ix) and head[COLUMN_KEY].reset_index(drop=True).equals(expect):
                        completed+=1
                        print('  CACHED',algorithm,set_name,flush=True)
                        continue
                except Exception as exc:
                    print('  RECOMPUTE corrupt cache:',repr(exc)[:150],flush=True)
            if MAX_NEW_FITS is not None and new_fits>=MAX_NEW_FITS:
                print('Reached fit cap; rerun Block 11B to resume.',flush=True)
                break
            xtr=D11.iloc[train_ix][cols].to_numpy('float32')
            xte=D11.iloc[test_ix][cols].to_numpy('float32')
            ytr=D11.iloc[train_ix].target_fraction.to_numpy('float32')
            model=estimator11(algorithm)
            start=time.monotonic()
            model.fit(xtr,ytr)
            pred=np.clip(model.predict(xte),0,1)
            saved=D11.iloc[test_ix][COLUMN_KEY+['group','x','y','target_fraction']].copy().reset_index(drop=True)
            saved=saved.rename(columns={'target_fraction':'observed'})
            saved['predicted']=pred
            tmp=path.with_suffix('.parquet.partial')
            saved.to_parquet(tmp,index=False)
            os.replace(tmp,path)
            new_fits+=1;completed+=1
            print('  SAVED',algorithm,set_name,'seconds',round(time.monotonic()-start,1),flush=True)
            del model,xtr,xte,ytr,saved
        if MAX_NEW_FITS is not None and new_fits>=MAX_NEW_FITS:
            break
    if MAX_NEW_FITS is not None and new_fits>=MAX_NEW_FITS:
        break
pd.DataFrame(fold_report).to_csv(BLOCK11/'11_fold_sizes.csv',index=False)
expected=[RUN11/(f'{scheme}_{heldout}_{name}'+(f'_{algorithm}' if algorithm!='HGB' else '')+'.parquet')
          for scheme,heldout in folds11 for algorithm in ALGORITHMS11 for name in FEATURE_SETS11]
missing=[path for path in expected if not path.exists()]
print('Checkpoint status:',len(expected)-len(missing),'/',len(expected),'model/fold results.')
if missing:
    print('INCOMPLETE — rerun this cell. Missing:',len(missing),'fits.')
else:
    all_rows=[]
    for scheme,heldout in folds11:
        for algorithm in ALGORITHMS11:
            for set_name in FEATURE_SETS11:
                stem=f'{scheme}_{heldout}_{set_name}'+(f'_{algorithm}' if algorithm!='HGB' else '')
                path=RUN11/f'{stem}.parquet'
                preds=pd.read_parquet(path)
                preds['scheme']=scheme;preds['holdout']=heldout
                preds['model']=set_name;preds['algorithm']=algorithm
                all_rows.append(preds)
    ALL11=pd.concat(all_rows,ignore_index=True)
    ALL11.to_parquet(BLOCK11/'11_all_heldout_predictions.parquet',index=False)
    by_site=[];by_fold=[];summary=[]
    for (scheme,algorithm,set_name),g in ALL11.groupby(['scheme','algorithm','model'],sort=False):
        # Across spatial folds, each target is held out once.
        # Across LOSO, each forest is held out once.
        assert not g.duplicated(COLUMN_KEY).any(),(scheme,set_name,'duplicated test predictions')
        by_fold.extend(dict(scheme=scheme,algorithm=algorithm,model=set_name,holdout=fold,
                            **metric11(sub.observed,sub.predicted))
                       for fold,sub in g.groupby('holdout'))
        site_stats=[]
        for site,sg in g.groupby('site'):
            site_stats.append(dict(scheme=scheme,algorithm=algorithm,model=set_name,site=site,
                                   **metric11(sg.observed,sg.predicted)))
        by_site.extend(site_stats)
        pooled=metric11(g.observed,g.predicted)
        macro_mae=np.mean([v['MAE_pp'] for v in site_stats])
        macro_rmse=np.mean([v['RMSE_pp'] for v in site_stats])
        macro_bias=np.mean([v['bias_pp'] for v in site_stats])
        macro_r2=np.nanmean([v['R2'] for v in site_stats])
        mos=g[g.site=='MOS'];mh=mos.observed>=HIGH_COVER
        # Equal-site weighted R², calculated on a common global target mean.
        counts=g.groupby('site').size()
        weights=1/g.site.map(counts).to_numpy(float);weights/=weights.sum()
        y=g.observed.to_numpy(float);prediction=g.predicted.to_numpy(float)
        avg=np.dot(weights,y)
        ss=np.dot(weights,(y-avg)**2)
        balanced_r2=1-np.dot(weights,(y-prediction)**2)/ss if ss>0 else np.nan
        summary.append(dict(scheme=scheme,algorithm=algorithm,model=set_name,n=len(g),
            pooled_R2=pooled['R2'],pooled_MAE_pp=pooled['MAE_pp'],
            pooled_RMSE_pp=pooled['RMSE_pp'],
            equal_forest_R2=float(balanced_r2),
            mean_site_R2=float(macro_r2),
            equal_forest_MAE_pp=float(macro_mae),
            equal_forest_RMSE_pp=float(macro_rmse),
            equal_forest_bias_pp=float(macro_bias),
            MOS_high_n=int(mh.sum()),
            MOS_high_bias_pp=float(100*(mos.predicted[mh]-mos.observed[mh]).mean()) if mh.any() else np.nan))
    SUMMARY11=pd.DataFrame(summary)
    SITES11=pd.DataFrame(by_site)
    FOLDS11=pd.DataFrame(by_fold)
    SUMMARY11.to_csv(BLOCK11/'11_global_metrics.csv',index=False)
    SITES11.to_csv(BLOCK11/'11_site_metrics.csv',index=False)
    FOLDS11.to_csv(BLOCK11/'11_fold_metrics.csv',index=False)
    for scheme in schemes11:
        print('VALIDATION:',scheme,'— compare predictors within each algorithm')
        display(SUMMARY11[SUMMARY11.scheme==scheme].sort_values('equal_forest_MAE_pp').round(3))
    print('Site-level diagnostics (LOSO):')
    display(SITES11[SITES11.scheme=='LEAVE_FOREST_OUT'].round(3))
    for scheme in schemes11:
        for algorithm in ALGORITHMS11:
            f=ALL11[(ALL11.scheme==scheme)&(ALL11.algorithm==algorithm)]
            fig,axes=plt.subplots(1,4,figsize=(16,4),layout='constrained')
            for ax,site in zip(axes,SITES):
                q=f[(f.site==site)&(f.model=='B0_S2_EMB')]
                q2=f[(f.site==site)&(f.model=='B5_B0_AllSpatial')]
                for sub,label in ((q,'B0'),(q2,'B5')):
                    ax.scatter(sub.observed*100,sub.predicted*100,s=2,
                               alpha=.12,label=label,rasterized=True)
                ax.plot([0,100],[0,100],ls='--',color='grey',lw=1)
                ax.set(xlim=(0,100),ylim=(0,100),title=site,
                       xlabel='UAV Musanga (%)',ylabel='Predicted (%)')
                ax.legend()
            fig.suptitle(f'{scheme} | {algorithm} | held-out B0 vs B5')
            fig.savefig(FIG11/f'{scheme}_{algorithm}_B0_vs_B5_scatter.png',dpi=150)
            plt.show()
            plt.close(fig)
    print('COMPLETE: all benchmark metrics and held-out predictions saved:',BLOCK11)


## 11C — HGB-only recovery of NB04 training strategy

Compare **two training samples (forest-balanced random vs NB04-style abundance-stratified)** × **two prediction variants (single HGB vs five independently resampled HGB models)** on only **B0 and B1**. Use the exact existing 11B spatial and leave-one-forest-out folds, no RF or Extra Trees fitting. The original random/single HGB predictions are reused unchanged as ensemble member 0; all existing 11B outputs stay untouched.

As in NB04, stratification reserves up to 25% of the training cap for examples with Musanga fraction ≥40%, when available, then approximately balances remaining draws among the four abundance bins. Each forest is capped separately at 1,500 cells, and no example is duplicated. Model hyperparameters remain frozen. The saved member predictions make this restartable; incomplete ensembles are never reported. Run 11A and 11B first in a fresh session (11B reuses existing predictions), then 11C.

Compare within **each feature set**: effect of sampling (random vs stratified), effect of averaging (single vs ensemble), and MOS high-cover bias versus Yangambi errors. This is a focused NB04 methodological comparison at 50 m, **not** a direct replication of NB04's 100 m scores.

In [ ]:
# 11C — NB04-style stratified HGB, five-member ensembles; restartable.
import os,hashlib,json,time
import numpy as np,pandas as pd
from scipy.spatial import cKDTree
from sklearn.ensemble import HistGradientBoostingRegressor
from IPython.display import display
assert all(k in globals() for k in ['D11','split11','RUN11','MODEL_ARGS','metric11','folds11'])
F11C=['B0_S2_EMB','B1_B0_PlanetSpectral']
N11C=5
MAX_NEW_FITS11C=None  # Set 3 to smoke-test; None runs remaining members.
conf11c=dict(version='11C_v1',parent=RUN11.name,features=F11C,
             folds=folds11,n_members=N11C,cap=TRAIN_CAP_PER_FOREST,
             seed=MODEL_SEED,model=MODEL_ARGS,high_share=.25)
key11c=hashlib.sha256(json.dumps(conf11c,sort_keys=True).encode()).hexdigest()[:16]
DIR11C=BLOCK11/'11C_HGB_sampling_ensemble'/key11c
DIR11C.mkdir(parents=True,exist_ok=True)
(DIR11C/'config.json').write_text(json.dumps(conf11c,indent=2))
sites11c=D11.site.to_numpy()
xy11c=D11[['x','y']].to_numpy(float)
y11c=D11.target_fraction.to_numpy(float)
def allowed11c(scheme,hold,test):
    eligible=np.ones(len(D11),dtype=bool)
    eligible[test]=False
    if scheme=='BLOCK_BUFFERED':
        for forest in SITES:
            ti=test[sites11c[test]==forest]
            ix=np.flatnonzero(eligible & (sites11c==forest))
            if len(ti) and len(ix):
                near=cKDTree(xy11c[ti]).query(xy11c[ix],k=1)[0]<BUFFER_M
                eligible[ix[near]]=False
    return np.flatnonzero(eligible)
def sample11c(pool,mode,seed):
    rng=np.random.default_rng(seed)
    chosen=[]
    for forest in SITES:
        ids=pool[sites11c[pool]==forest]
        cap=min(len(ids),TRAIN_CAP_PER_FOREST)
        if len(ids)<=cap:
            chosen.extend(ids.tolist());continue
        if mode=='random':
            selected=rng.choice(ids,cap,replace=False).tolist()
        else:
            classes=np.digitize(y11c[ids],[.10,.20,.40])
            bins={k:ids[classes==k].copy() for k in range(4)}
            n_high=min(len(bins[3]),int(np.ceil(.25*cap)))
            selected=rng.choice(bins[3],n_high,replace=False).tolist()
            bins[3]=np.setdiff1d(bins[3],np.asarray(selected,dtype=int))
            while len(selected)<cap:
                active=[k for k in range(4) if len(bins[k])]
                assert active
                quota=max(1,(cap-len(selected))//len(active))
                for k in active:
                    n=min(quota,len(bins[k]),cap-len(selected))
                    if n:
                        new=rng.choice(bins[k],n,replace=False)
                        selected.extend(new.tolist())
                        bins[k]=np.setdiff1d(bins[k],new)
                    if len(selected)==cap:break
        assert len(selected)==cap and len(set(selected))==cap
        chosen.extend(selected)
    return np.asarray(sorted(chosen),dtype=int)
def archived11c(scheme,hold,feature):
    return RUN11/f'{scheme}_{hold}_{feature}.parquet'
def checkpoint11c(scheme,hold,feature,mode,member):
    return (archived11c(scheme,hold,feature) if mode=='random' and member==0
            else DIR11C/f'{scheme}_{hold}_{feature}_{mode}_m{member}.parquet')
def valid11c(path,expected):
    if not path.exists():return False
    try:
        d=pd.read_parquet(path)
        return (len(d)==len(expected) and d[COLUMN_KEY].reset_index(drop=True).equals(expected)
                and np.isfinite(d.predicted).all())
    except Exception:return False
new11c=0;stop11c=False
for scheme,hold in folds11:
    train_original,test,_=split11(scheme,hold)
    pool=allowed11c(scheme,hold,test)
    # Reconstruct original 11B sampling exactly. Its random member 0 is cached.
    assert np.array_equal(sample11c(pool,'random',MODEL_SEED),train_original)
    expected=D11.iloc[test][COLUMN_KEY].reset_index(drop=True)
    for feature in F11C:
        assert valid11c(archived11c(scheme,hold,feature),expected)
        for mode in ('random','stratified'):
            for member in range(N11C):
                path=checkpoint11c(scheme,hold,feature,mode,member)
                if valid11c(path,expected):continue
                if MAX_NEW_FITS11C is not None and new11c>=MAX_NEW_FITS11C:
                    stop11c=True;break
                seed=MODEL_SEED+1009*member+sum(map(ord,str(hold)))+(100000 if mode=='stratified' else 0)
                ids=sample11c(pool,mode,seed)
                kwargs=dict(MODEL_ARGS,random_state=seed)
                start=time.monotonic()
                model=HistGradientBoostingRegressor(**kwargs)
                model.fit(D11.iloc[ids][FEATURE_SETS11[feature]].to_numpy('float32'),
                          y11c[ids].astype('float32'))
                pred=np.clip(model.predict(D11.iloc[test][FEATURE_SETS11[feature]].to_numpy('float32')),0,1)
                output=D11.iloc[test][COLUMN_KEY+['target_fraction']].copy().reset_index(drop=True)
                output=output.rename(columns={'target_fraction':'observed'})
                output['predicted']=pred
                tmp=path.with_suffix('.parquet.partial')
                output.to_parquet(tmp,index=False);os.replace(tmp,path)
                new11c+=1
                print('SAVED',scheme,hold,feature,mode,'member',member,
                      round(time.monotonic()-start,1),'s',flush=True)
            if stop11c:break
        if stop11c:break
    if stop11c:break
tasks11c=[(sc,ho,fe,mo,me) for sc,ho in folds11 for fe in F11C
          for mo in ('random','stratified') for me in range(N11C)]
missing11c=[key for key in tasks11c
            if not valid11c(checkpoint11c(*key),
                D11.iloc[split11(key[0],key[1])[1]][COLUMN_KEY].reset_index(drop=True))]
print('11C MEMBER CACHE:',len(tasks11c)-len(missing11c),'/',len(tasks11c),flush=True)
if missing11c:
    print('PARTIAL — rerun 11C to resume; outstanding',len(missing11c),flush=True)
else:
    reports=[];site_reports=[];prediction_frames=[]
    for scheme,hold in folds11:
        for feature in F11C:
            for mode in ('random','stratified'):
                members=[pd.read_parquet(checkpoint11c(scheme,hold,feature,member_mode,idx))
                         for member_mode,idx in [(mode,i) for i in range(N11C)]]
                assert all(x[COLUMN_KEY].equals(members[0][COLUMN_KEY]) for x in members)
                mat=np.stack([x.predicted.to_numpy(float) for x in members])
                for variant,pred in [('single',mat[0]),('ensemble5',mat.mean(axis=0))]:
                    q=members[0][COLUMN_KEY+['observed']].copy()
                    q['predicted']=np.clip(pred,0,1)
                    q['scheme']=scheme;q['holdout']=hold
                    q['feature']=feature;q['sampling']=mode;q['variant']=variant
                    prediction_frames.append(q)
    P11C=pd.concat(prediction_frames,ignore_index=True)
    P11C.to_parquet(DIR11C/'heldout_predictions.parquet',index=False)
    for (scheme,feature,sampling,variant),g in P11C.groupby(['scheme','feature','sampling','variant']):
        assert not g.duplicated(COLUMN_KEY).any()
        site_metrics=[]
        for site,t in g.groupby('site'):
            x=dict(scheme=scheme,feature=feature,sampling=sampling,
                   variant=variant,site=site,**metric11(t.observed,t.predicted))
            site_metrics.append(x);site_reports.append(x)
        counts=g.groupby('site').size()
        weights=1/g.site.map(counts).to_numpy(float);weights/=weights.sum()
        y=g.observed.to_numpy(float);p=g.predicted.to_numpy(float)
        mu=np.dot(weights,y);denom=np.dot(weights,(y-mu)**2)
        mos=g[g.site=='MOS'];hi=mos.observed>=.40
        reports.append(dict(scheme=scheme,feature=feature,sampling=sampling,
            variant=variant,n=len(g),pooled_R2=metric11(y,p)['R2'],
            equal_forest_R2=1-np.dot(weights,(y-p)**2)/denom,
            equal_forest_MAE_pp=np.mean([x['MAE_pp'] for x in site_metrics]),
            mean_site_R2=np.nanmean([x['R2'] for x in site_metrics]),
            MOS_high_n=int(hi.sum()),MOS_high_bias_pp=float(
                100*(mos.predicted[hi]-mos.observed[hi]).mean()) if hi.any() else np.nan))
    results11c=pd.DataFrame(reports);by_site11c=pd.DataFrame(site_reports)
    results11c.to_csv(DIR11C/'global_metrics.csv',index=False)
    by_site11c.to_csv(DIR11C/'site_metrics.csv',index=False)
    for scheme in ('BLOCK_BUFFERED','LEAVE_FOREST_OUT'):
        print('11C',scheme)
        display(results11c[results11c.scheme==scheme].sort_values(
            ['feature','equal_forest_MAE_pp']).round(3))
    print('COMPLETE:',DIR11C,flush=True)


## 11D — MOS prediction audit (no model fitting)

Investigate why the **leave-one-forest-out MOS high-abundance bias** in 11C appears exactly identical for random, stratified, single and ensemble HGB models **within each feature set**. Read **actual saved held-out predictions**, and independently recompute high-cover bias. Do **not** assume a reporting bug before inspecting the prediction vectors.

The audit checks (a) whether predictions vary at all, (b) whether predictions vary specifically for MOS observations ≥40% Musanga, (c) where predictions saturate/flatten, (d) potential leakage or stale results through unchanged predictions, and (e) changes within abundance bins and MOS mosaics. The diagnostic also verifies random/single predictions against archived 11B files and the arithmetic mean of the five member predictions. Includes high-resolution numeric tables and held-out scatter/calibration plots.

**Critical distinction:** identical *bias* can arise from different predictions that have the same mean. Conversely, precisely identical bias alongside nearly constant predictions may indicate saturation. Neither should be inferred from aggregate metrics alone.

Only persisted 11C predictions are necessary. Run initialization (Section 00), then this block; it auto-detects the completed 11C run under the usual NB06 output path. No source UAV/Planet mosaics are read or recomputed.

In [ ]:
# 11D — Independent audit of cached 11C predictions, focused on held-out MOS.
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from IPython.display import display
from itertools import combinations
import hashlib

ROOT11D=(OUT/'11_spatial_ablation_50m'/'11C_HGB_sampling_ensemble')
folders11d=sorted((p for p in ROOT11D.glob('*') if (p/'heldout_predictions.parquet').exists()),
                  key=lambda p:(p/'heldout_predictions.parquet').stat().st_mtime,reverse=True)
assert folders11d,'No completed 11C heldout_predictions.parquet. Run 11C first.'
DIR11D=folders11d[0]
AUDIT11D=DIR11D/'11D_diagnostics'
AUDIT11D.mkdir(exist_ok=True)
print('Auditing saved 11C results:',DIR11D,flush=True)
P=pd.read_parquet(DIR11D/'heldout_predictions.parquet')
need={'site','mosaic','row','col','observed','predicted',
      'scheme','holdout','feature','sampling','variant'}
assert need.issubset(P.columns),sorted(need-set(P.columns))
assert np.isfinite(P[['observed','predicted']].to_numpy(float)).all()
KEYS11D=['site','mosaic','row','col']
P['configuration']=P['feature']+' | '+P['sampling']+' | '+P['variant']
F11D=['B0_S2_EMB','B1_B0_PlanetSpectral']
MODES11D=[('random','single'),('random','ensemble5'),
          ('stratified','single'),('stratified','ensemble5')]
MOS11D=P[(P.scheme=='LEAVE_FOREST_OUT')&(P.site=='MOS')].copy()
assert MOS11D.holdout.eq('MOS').all(),'MOS data appears outside MOS holdout!'
expected11d=len(MOS11D[MOS11D.configuration==MOS11D.configuration.iloc[0]])
print('MOS test cells per configuration:',expected11d,
      '| high >=40%:',int((MOS11D[MOS11D.configuration==MOS11D.configuration.iloc[0]].observed>=.4).sum()))
assert expected11d>1000
assert len(MOS11D)==expected11d*len(F11D)*len(MODES11D)
# Every model must predict the exact same MOS cells and UAV targets.
mosaics11d={}
for feature in F11D:
    for mode,variant in MODES11D:
        q=MOS11D[(MOS11D.feature==feature)&(MOS11D.sampling==mode)&
                 (MOS11D.variant==variant)].sort_values(KEYS11D).reset_index(drop=True)
        assert len(q)==expected11d and not q.duplicated(KEYS11D).any()
        mosaics11d[(feature,mode,variant)]=q
base11d=mosaics11d[(F11D[0],'random','single')]
for q in mosaics11d.values():
    assert q[KEYS11D].equals(base11d[KEYS11D]),'Mismatched held-out MOS cell coordinates'
    assert np.allclose(q.observed,base11d.observed,atol=1e-8),'Different UAV targets!'
    
# Independently recompute MOS high-cover statistics (unrounded; by mosaic too).
high11d=base11d.observed.to_numpy(float)>=.40
y11d=base11d.observed.to_numpy(float)
stats11d=[]
for (feature,mode,variant),q in mosaics11d.items():
    pred=q.predicted.to_numpy(float)
    err=(pred-y11d)*100
    hi=pred[high11d]
    stats11d.append(dict(feature=feature,sampling=mode,variant=variant,
        n=len(q),n_high=int(high11d.sum()),
        high_actual_mean_pp=100*float(y11d[high11d].mean()),
        high_pred_mean_pp=100*float(hi.mean()),
        high_bias_pp=float(err[high11d].mean()),
        high_MAE_pp=float(abs(err[high11d]).mean()),
        high_RMSE_pp=float(np.sqrt(np.mean(err[high11d]**2))),
        high_recall_40=float((hi>=.40).mean()),
        high_pred_SD_pp=100*float(hi.std()),
        high_pred_min_pp=100*float(hi.min()),
        high_pred_p50_pp=100*float(np.median(hi)),
        high_pred_max_pp=100*float(hi.max()),
        high_unique_6dp=int(len(np.unique(np.round(hi,6)))),
        high_pred_sha256=hashlib.sha256(hi.tobytes()).hexdigest()[:12]))
S11D=pd.DataFrame(stats11d)
S11D.to_csv(AUDIT11D/'11D_MOS_high_detailed.csv',index=False,float_format='%.12g')
print('MOS >=40%: independent, UNROUNDED metrics and prediction-vector fingerprints')
display(S11D)

# Pairwise matched predictions: compare both whole MOS and its high-cover tail.
changes11d=[]
for feature in F11D:
    candidates=[(mode,variant) for mode,variant in MODES11D]
    for (m1,v1),(m2,v2) in combinations(candidates,2):
        a=mosaics11d[(feature,m1,v1)].predicted.to_numpy(float)
        b=mosaics11d[(feature,m2,v2)].predicted.to_numpy(float)
        dif=100*(b-a)
        for label,mask in [('ALL',np.ones(len(a),dtype=bool)),('HIGH_GE40',high11d)]:
            d=dif[mask]
            changes11d.append(dict(feature=feature,first=m1+'_'+v1,second=m2+'_'+v2,
                subset=label,n=int(mask.sum()),mean_delta_pp=float(d.mean()),
                mean_abs_delta_pp=float(abs(d).mean()),
                max_abs_delta_pp=float(abs(d).max()),
                pct_bitwise_identical=float(100*np.mean(a[mask]==b[mask])),
                pct_within_0_01pp=float(100*np.mean(abs(d)<.01))))
CHANGES11D=pd.DataFrame(changes11d)
CHANGES11D.to_csv(AUDIT11D/'11D_pairwise_prediction_differences.csv',index=False)
print('Paired prediction changes, including all high-Musanga pixels:')
display(CHANGES11D.round(5))

# All predicted/observed abundance classes; per mosaic confirms MOS subdomains.
bins11d=[-.000001,.10,.20,.40,.60,.80,1.000001]
labels11d=['0–10','10–20','20–40','40–60','60–80','80–100']
binned11d=[]
for (feature,mode,variant),q in mosaics11d.items():
    tmp=q.copy()
    tmp['abundance_bin']=pd.cut(tmp.observed,bins11d,labels=labels11d)
    for (mosaic,bname),chunk in tmp.groupby(['mosaic','abundance_bin'],observed=True):
        err=100*(chunk.predicted-chunk.observed)
        binned11d.append(dict(feature=feature,sampling=mode,variant=variant,
            mosaic=mosaic,observed_bin=str(bname),n=len(chunk),
            observed_mean_pp=float(100*chunk.observed.mean()),
            predicted_mean_pp=float(100*chunk.predicted.mean()),
            bias_pp=float(err.mean()),MAE_pp=float(abs(err).mean()),
            predicted_sd_pp=float(100*chunk.predicted.std(ddof=0)),
            fraction_pred_ge40=float((chunk.predicted>=.40).mean())))
BINS11D=pd.DataFrame(binned11d)
BINS11D.to_csv(AUDIT11D/'11D_MOS_by_mosaic_abundance_bin.csv',index=False)
print('MOS: by mosaic and observed Musanga abundance bin')
display(BINS11D.round(3))

# Verify saved ensemble averages and original random/single checkpoint for MOS.
# Only read 11B predictions if available, no model refitting.
FOLD_ROOT11D=OUT/'11_spatial_ablation_50m'/'fold_predictions'
parent11d=(DIR11D/'config.json')
if parent11d.exists():
    import json
    config11d=json.loads(parent11d.read_text())
    run_name11d=config11d.get('parent')
    ORIGINAL11D=FOLD_ROOT11D/run_name11d if run_name11d else None
else:
    ORIGINAL11D=None
checks11d=[]
for feature in F11D:
    first=mosaics11d[(feature,'random','single')]
    path=(ORIGINAL11D/f'LEAVE_FOREST_OUT_MOS_{feature}.parquet') if ORIGINAL11D else None
    if path is not None and path.exists():
        ref=pd.read_parquet(path).sort_values(KEYS11D).reset_index(drop=True)
        assert ref[KEYS11D].equals(first[KEYS11D])
        checks11d.append(dict(feature=feature,test='11B_vs_11C_random_single',
                              max_abs_diff_pp=float(100*abs(
                                  ref.predicted.to_numpy()-first.predicted.to_numpy()).max())))
    for mode in ('random','stratified'):
        member_files=[]
        for member in range(5):
            fn=(ORIGINAL11D/f'LEAVE_FOREST_OUT_MOS_{feature}.parquet'
                if mode=='random' and member==0 else
                DIR11D/f'LEAVE_FOREST_OUT_MOS_{feature}_{mode}_m{member}.parquet')
            assert fn is not None and fn.exists(),f'Missing ensemble member: {fn}'
            member_files.append(pd.read_parquet(fn).sort_values(KEYS11D).reset_index(drop=True))
        assert all(f[KEYS11D].equals(first[KEYS11D]) for f in member_files)
        mean_pred=np.stack([f.predicted.to_numpy(float) for f in member_files]).mean(axis=0)
        ensemble=mosaics11d[(feature,mode,'ensemble5')].predicted.to_numpy(float)
        checks11d.append(dict(feature=feature,test=mode+'_five_member_average',
                              max_abs_diff_pp=float(100*np.max(abs(mean_pred-ensemble)))))
CHECKS11D=pd.DataFrame(checks11d)
CHECKS11D.to_csv(AUDIT11D/'11D_cache_integrity_checks.csv',index=False)
display(CHECKS11D)
assert (CHECKS11D.max_abs_diff_pp<1e-5).all(),'11C aggregation does not match its constituent predictions!'
print('Checkpoint and ensemble-integrity checks passed.')

# Side-by-side held-out MOS plots; low-alpha thousands of points remain readable.
fig,axs=plt.subplots(2,3,figsize=(15,9),constrained_layout=True)
for ix,feature in enumerate(F11D):
    ax=axs[ix,0]
    for mode,variant in MODES11D:
        q=mosaics11d[(feature,mode,variant)]
        name=mode[:5]+'_'+variant
        ax.scatter(q.observed*100,q.predicted*100,s=2,alpha=.08,rasterized=True,label=name)
    ax.plot([0,100],[0,100],'--',color='grey',lw=1)
    ax.axvline(40,ls=':',color='grey',lw=1)
    ax.set(xlim=(0,100),ylim=(0,100),title=feature+' | held-out MOS',
           xlabel='Observed UAV Musanga (%)',ylabel='Predicted satellite (%)')
    ax.legend(fontsize=7,markerscale=3)
    ax=axs[ix,1]
    for mode,variant in MODES11D:
        q=mosaics11d[(feature,mode,variant)]
        b=pd.cut(q.observed,bins11d,labels=labels11d)
        means=q.groupby(b,observed=True).predicted.mean()
        ax.plot([5,15,30,50,70,90][:len(means)],means.values*100,
                marker='o',label=mode[:5]+'_'+variant)
    ax.plot([0,100],[0,100],'--',color='grey',lw=1)
    ax.set(xlim=(0,100),ylim=(0,100),title='Mean predicted by observed bin',
           xlabel='Observed Musanga class midpoint (%)',ylabel='Mean predicted (%)')
    ax.legend(fontsize=7)
    ax=axs[ix,2]
    for mode,variant in MODES11D:
        q=mosaics11d[(feature,mode,variant)]
        hi=q.loc[q.observed>=.4,'predicted']*100
        ax.hist(hi,bins=np.linspace(0,100,26),histtype='step',density=True,
                lw=1.3,label=mode[:5]+'_'+variant)
    ax.axvline(40,ls=':',color='grey',lw=1)
    ax.set(title='Predictions for observed >=40%',
           xlabel='Predicted Musanga (%)',ylabel='Probability density')
    ax.legend(fontsize=7)
fig.savefig(AUDIT11D/'11D_MOS_high_abundance_diagnostics.png',dpi=160)
plt.show();plt.close(fig)

# Make audit conclusion explicit without assuming explanation in advance.
print('INDEPENDENT AUDIT INTERPRETATION:')
for feature,part in S11D.groupby('feature'):
    spread=float(part.high_bias_pp.max()-part.high_bias_pp.min())
    pair=CHANGES11D[(CHANGES11D.feature==feature)&(CHANGES11D.subset=='HIGH_GE40')]
    biggest=float(pair.max_abs_delta_pp.max())
    print(feature,'high-cover bias spread (pp):',f'{spread:.12g}',
          '| largest per-cell prediction difference (pp):',f'{biggest:.12g}')
    if spread<1e-9 and biggest>1e-5:
        print('  Same MEAN high-cover bias but different predictions: inspect saturation / cancellation.')
    elif spread<1e-9 and biggest<=1e-5:
        print('  Nearly identical high-cover predictions: inspect saturation or shared predictor limits.')
    else:
        print('  High-cover bias genuinely differs; the previous displayed equality requires a reporting check.')
print('11D saved CSVs and figure to:',AUDIT11D)


## 12 · Native PlanetScope patch CNN — matched geographical validation

**12A (one-time, restartable):** extract *actual* 4.77 m Planet RGB+NIR patches (four native reflectance bands B,G,R,N, 32 × 32 pixels ≈153 m × 153 m) centered on all 14,454 accepted 50 m UAV reference cells. No 50 m Planet raster is produced and no UAV full mosaic is reopened. Patches have ~153 m spatial context around a **50 m fractional abundance target**; a patch is *not* a pixel-level Musanga map. Cache 200-patch batches per site on Drive with SHA-based source/target settings, and make subsequent runs skip verified complete chunks. Input output stores native 4-band image tensors as float16 and pixel-validity QA separately. Native 32×32 images are preserved, rather than reduced to means/SDs.

**12B (restartable GPU CNN):** supervised 4-channel CNN → one *50 m Musanga fraction*, with the **same exact 7 test folds and forest-balanced training-cell selection** as Block 11. Each of the three 400 m block tests has a 150 m label-center exclusion buffer; LOFO holds out each entire forest (including both MOS mosaics). Training-only per-band robust normalization; training-only flip/rotation augmentation. Fixed training epochs and architecture (no model selection based on the test forest). Separate checkpoint per fold; saved predictions are matched one-to-one to 11B HGB B0 and B1 held-out predictions. Report equal-forest R²/MAE, pooled R² and **MOS ≥40% high-abundance bias/recall**, plus per-site results and scatterplots.

**What this isolates:** does learning *native Planet spatial patterns* carry predictive signal without handcrafted texture features? The CNN uses Planet-only inputs, whereas HGB B1 uses S2, embeddings, and Planet means/SDs, so a direct superiority claim would conflate the data sources. Later test a **hybrid** model (CNN representation + S2/embeddings), trained and assessed on the same folds, if the Planet patch CNN looks promising.

**Efficiency:** 12A uses small raster windows and caches all native patches; 12B loads ~120 MB of patches, fits serially on GPU if available, prints per-epoch loss and writes each fold's predictions atomically. Use `MAX_NEW_BATCHES12A=2` for extraction smoke test, or `MAX_NEW_FOLDS12B=1` for training smoke test. No prior sections/models are retrained.

**Caveats:** patch neighborhoods can include satellite pixels across nearby fold boundaries (only imagery, not labels); the 150 m exclusion buffer is applied to labeled cell centers. Some Planet epochs and UAV acquisition years differ (particularly Yoko). Success is judged on *held-out UAV pseudo-label fractions*, not field-measured Musanga or individual-crown resolution.


In [ ]:
# 12A — Native 4-band Planet 32x32 patch extraction; per-site restartable.
from pathlib import Path
import os,json,hashlib,time,gc
import numpy as np,pandas as pd,rasterio
from rasterio.windows import Window
from pyproj import Transformer
from IPython.display import display
assert 'OUT' in globals() and 'SITES' in globals() and 'planet_file' in globals()
P12=32
CHUNK12=200
MAX_NEW_BATCHES12A=None  # Try 2 to check extraction; None finishes all.
OUT12=OUT/'12_native_planet_cnn'
CACHE12=OUT12/'native_patches'
CACHE12.mkdir(parents=True,exist_ok=True)
FILE12=OUT/'10_full_spatial_features_50m.parquet'
assert FILE12.exists(),FILE12
K12=['site','mosaic','row','col']
T12=pd.read_parquet(FILE12,columns=K12+['x','y','crs','target_fraction'])
T12=T12.sort_values(K12,kind='stable').reset_index(drop=True)
assert not T12.duplicated(K12).any()
assert len(T12)==14454
T12['global_index']=np.arange(len(T12),dtype=np.int32)
PLANET_PATH12={site:planet_file(site,'Annual') for site in SITES}
manifest12=dict(version='P12_32x32_BGRN_v1',size=P12,chunk=CHUNK12,bands=[13,14,15,16],
                target_digest=hashlib.sha256(pd.util.hash_pandas_object(
                    T12[K12+['x','y','crs','target_fraction']],index=False
                ).values.tobytes()).hexdigest(),
                sources={site:dict(path=str(path),size=path.stat().st_size,
                    mtime_ns=path.stat().st_mtime_ns) for site,path in PLANET_PATH12.items()})
id12=hashlib.sha256(json.dumps(manifest12,sort_keys=True).encode()).hexdigest()[:16]
RUN12=CACHE12/id12
RUN12.mkdir(parents=True,exist_ok=True)
(RUN12/'config.json').write_text(json.dumps(manifest12,indent=2,sort_keys=True))
print('Native Planet patches:',len(T12),'| cache:',RUN12,flush=True)
new12=0
for site in SITES:
    sub=T12[T12.site==site].reset_index(drop=True)
    site_dir=RUN12/site;site_dir.mkdir(exist_ok=True)
    # Each chunk is independent; a completed chunk is never overwritten.
    with rasterio.open(PLANET_PATH12[site]) as ds:
        assert ds.count>=16 and ds.crs and abs(ds.res[0]-4.77)<.5,(site,ds.res)
        transformer=Transformer.from_crs(sub.crs.iloc[0],ds.crs,always_xy=True)
        assert sub.crs.nunique()==1,(site,'mixed CRS')
        for off in range(0,len(sub),CHUNK12):
            part=sub.iloc[off:off+CHUNK12]
            dest=site_dir/f'patches_{off:06d}_{len(part):03d}.npz'
            if dest.exists():
                try:
                    with np.load(dest) as pack:
                        assert pack['patches'].shape==(len(part),4,P12,P12)
                        assert np.array_equal(pack['index'],part.global_index.to_numpy())
                        assert pack['valid'].shape==(len(part),)
                    print('CACHED',site,dest.name,flush=True)
                    continue
                except Exception as exc:
                    print('REBUILD incomplete patch chunk',dest.name,repr(exc)[:120],flush=True)
            if MAX_NEW_BATCHES12A is not None and new12>=MAX_NEW_BATCHES12A:
                print('Reached extraction batch limit; rerun 12A to continue.',flush=True)
                break
            started=time.monotonic()
            patches=np.zeros((len(part),4,P12,P12),dtype=np.float16)
            valid=np.zeros(len(part),dtype=np.float32)
            for j,rec in enumerate(part.itertuples(index=False)):
                xx,yy=transformer.transform(rec.x,rec.y)
                cc,rr=(~ds.transform)*(xx,yy)
                c=int(np.floor(cc));r=int(np.floor(rr))
                data=ds.read([13,14,15,16],window=Window(
                    c-P12//2,r-P12//2,P12,P12),boundless=True,fill_value=0,
                    out_dtype='float32')
                good=np.isfinite(data).all(axis=0)&(data>0).all(axis=0)
                valid[j]=float(good.mean())
                data[:,~good]=0
                patches[j]=np.clip(data,-65000,65000).astype(np.float16)
            tmp=dest.with_suffix('.npz.partial')
            # Write file object explicitly to avoid np.savez adding '.npz' suffix.
            with open(tmp,'wb') as stream:
                np.savez_compressed(stream,patches=patches,
                                    index=part.global_index.to_numpy(dtype='int32'),
                                    valid=valid)
            os.replace(tmp,dest)
            new12+=1
            print('SAVED',site,dest.name,'valid>=80%',int((valid>=.80).sum()),
                  '/',len(valid),'seconds',round(time.monotonic()-started,1),flush=True)
        if MAX_NEW_BATCHES12A is not None and new12>=MAX_NEW_BATCHES12A:
            break
# Consolidation only after ALL expected site chunks are present.
missing12=[]
for site in SITES:
    sub=T12[T12.site==site]
    for off in range(0,len(sub),CHUNK12):
        part=sub.iloc[off:off+CHUNK12]
        path=RUN12/site/f'patches_{off:06d}_{len(part):03d}.npz'
        if not path.exists():missing12.append(str(path))
print('12A chunks remaining:',len(missing12),flush=True)
if not missing12:
    # One compact, sequentially ordered tensor for fast training.
    # float16 array is 118 MB; store as .npy for memory mapping.
    tensor_path=RUN12/'planet_native_32px_float16.npy'
    validity_path=RUN12/'patch_validity.parquet'
    from numpy.lib.format import open_memmap
    X12disk=open_memmap(tensor_path,mode='w+',dtype='float16',
                        shape=(len(T12),4,P12,P12))
    valid_all=np.full(len(T12),np.nan,dtype='float32')
    for site in SITES:
        sub=T12[T12.site==site].reset_index(drop=True)
        for off in range(0,len(sub),CHUNK12):
            part=sub.iloc[off:off+CHUNK12]
            path=RUN12/site/f'patches_{off:06d}_{len(part):03d}.npz'
            with np.load(path) as pack:
                indices=pack['index']
                assert np.array_equal(indices,part.global_index.to_numpy())
                X12disk[indices]=pack['patches']
                valid_all[indices]=pack['valid']
    X12disk.flush();del X12disk
    assert np.isfinite(valid_all).all()
    QA12=T12[K12+['global_index','target_fraction']].copy()
    QA12['planet_native_valid']=valid_all
    QA12.to_parquet(validity_path,index=False)
    display(QA12.assign(good=QA12.planet_native_valid>=.8).groupby('site').agg(
        total=('good','size'),valid=('good','sum'),
        mean_valid=('planet_native_valid','mean')).round(3))
    print('12A COMPLETE:',tensor_path,'| QA:',validity_path,flush=True)
else:
    print('12A incomplete; rerun this block (saved chunks are reused).',flush=True)
